# WESAD preprocesamiento

Este notebook implementa el pipeline completo de preprocesamiento del dataset WESAD para la construcción de `StressLoad(t)`, desde los ficheros crudos del Empatica E4 hasta las matrices de características listas para entrenar el clasificador de estrés. Se organiza en tres bloques principales.

El **bloque 0** define el andamiaje del pipeline: configuración de rutas externas, y las funciones auxiliares reutilizables que se invocarán después para cada sujeto — extracción de características en ventanas móviles, orquestación completa por sujeto, normalización por baseline y conversión de timestamps.

El **bloque 1** construye el dataset a resolución de 1Hz. Primero se desarrolla y valida el pipeline sobre un único sujeto de prueba (S7), recorriendo paso a paso la carga de señales, la unificación en un dataframe común, la incorporación de las etiquetas de estado desde el `.pkl` de WESAD y la integración del IBI. Una vez validada la lógica sobre S7, se extiende automáticamente a todos los sujetos del dataset mediante el pipeline global.

El **bloque 2** agrega el dataset anterior a resolución de 1 minuto (1/60Hz), la granularidad efectivamente utilizada para el modelado de `StressLoad(t)`, y cierra con una auditoría de calidad que verifica la integridad estructural de la matriz maestra, la corrección matemática de la normalización Z-score y la coherencia temporal de las variables de lag.

# 0. Instalación y funciones auxiliares

## 0.1. Instalacion y imports de elementos necesarios  
Instalaciones trasladadas al notebook Instalaciones. 
Imports a continuación:

In [4]:
import pandas as pd
import numpy as np
import datetime
import pickle
from pathlib import Path
import json


## 0.2. Funciones auxiliares

En este apartado se definen algunas funciones auxiliares que se utilizan posteriormente a lo largo del código

### 0.2.1. Carga del mapa de rutas externo
Esta subsección externaliza las rutas a los ficheros crudos de WESAD a un fichero de configuración (WESAD_config_rutas.json), cargado al inicio y validado con un control de existencia explícito. Esto desacopla el código del pipeline del entorno de ejecución concreto, favoreciendo la reproducibilidad frente a rutas absolutas embebidas en el notebook.

In [5]:
# Carga del mapa de rutas externo
# Se lee el mapa de rutas externo
ruta_config = Path("WESAD_config_rutas.json")
if not ruta_config.exists():
    raise FileNotFoundError("⚠️ No se encontró 'WESAD_config_rutas.json'. Asegúrate de que esté en la misma carpeta que este notebook.")

with open(ruta_config, "r", encoding="utf-8") as f:
    config = json.load(f)

### 0.2.2. extract_physiological_features(df_unido_1hz, df_ibi, sampling_mode='1hz')
Esta función constituye el núcleo de ingeniería de características para `StressLoad(t)`: a partir de las señales fisiológicas unificadas a 1Hz y ya etiquetadas, calcula ventanas móviles de 1 minuto sobre HR, EDA, TEMP y ACC, junto con los estadísticos de variabilidad cardíaca (RMSSD, SDNN) derivados de los intervalos IBI crudos. Para cada instante *t* de la rejilla temporal se conserva tanto el valor instantáneo de cada señal como su agregado sobre la ventana retrospectiva de 60 segundos, y se colapsa la etiqueta de estado experimental correspondiente, marcando además si la ventana es "pura" (un único estado) o mixta (transición entre estados).

Una decisión de eficiencia computacional relevante es el parámetro `hrv_stride_s`: en modo `'1hz'`, el RMSSD/SDNN no se recalcula en cada segundo sino cada `hrv_stride_s` segundos (por defecto, 30), manteniendo el último valor válido en los pasos intermedios. Esto reduce significativamente el coste del bucle sin alterar la resolución efectiva de la señal de HRV, cuya dinámica es intrínsecamente más lenta que la de HR o EDA.

In [6]:
def extract_physiological_features(df_unido_1hz, df_ibi, sampling_mode='1hz', hrv_stride_s=30): # hrv_stride_s: cada cuántos segundos se recalcula RMSSD/SDNN.
    """
    Pipeline de ingeniería de características para StressLoad(t).
    Recibe el df unificado a 1Hz YA ETIQUETADO y calcula las ventanas móviles.
    """
    t_start = max(df_unido_1hz['Tiempo'].min(), df_ibi['Tiempo'].min()).floor('s')
    t_end = min(df_unido_1hz['Tiempo'].max(), df_ibi['Tiempo'].max()).ceil('s')
    
    if sampling_mode == '1hz':
        step_delta = pd.Timedelta(seconds=1)
    elif sampling_mode == '1min':
        step_delta = pd.Timedelta(minutes=1)
    else:
        raise ValueError("Modo no soportado. Elige '1hz' o '1min'.")
        
    window_duration = pd.Timedelta(minutes=1)
    grid_tiempo = pd.date_range(start=t_start + window_duration, end=t_end, freq=step_delta)
    
    # Índices optimizados
    df_ibi_idx = df_ibi.set_index('Tiempo').sort_index()
    df_m_idx = df_unido_1hz.set_index('Tiempo').sort_index()
    
    features_list = []
    print(f"🚀 Extrayendo características y colapsando etiquetas en modo [{sampling_mode}]...")

    last_rmssd, last_sdnn = np.nan, np.nan # inicializacion de variables donde "guardamos" el último RMSSD/SDNN calculado.

    # En vez de "for t in grid_tiempo", se usa enumerate
    # para tener también "i", que es un contador de pasos (0, 1, 2, 3...).
    # Como cada paso del bucle avanza 1 segundo (step_delta = 1s en modo '1hz'),
    # "i" es literalmente "cuántos segundos han pasado desde el inicio de la serie"
    
    for i, t in enumerate(grid_tiempo):
        window_start = t - window_duration
        # "i % hrv_stride_s == 0" quiere decir "el resto de dividir i entre hrv_stride_s es 0", es decir, "cada hrv_stride_s segundos" hay que recomputar.
        recomputar_hrv = (sampling_mode != '1hz') or (i % hrv_stride_s == 0) 

        # --- HRV (Sobre IBI crudo) ---
        if recomputar_hrv:
            # Este bloque ahora solo se ejecuta 1 de cada hrv_stride_s veces, no en cada vuelta del bucle principal.
            v_ibi = df_ibi_idx.loc[window_start:t, 'Intervalo_ms'].values

            # Identificación de si la ventana tiene cobertura real de IBI (es decir, si hay datos de IBI en la ventana)
            ventana_con_cobertura_real = (
                not df_ibi_idx.empty
                and df_ibi_idx.index.min() <= window_start
            )

            if len(v_ibi) > 5 and ventana_con_cobertura_real:
                last_rmssd = np.sqrt(np.mean(np.diff(v_ibi) ** 2))
                last_sdnn = np.std(v_ibi, ddof=1)
            else:
                last_rmssd, last_sdnn = np.nan, np.nan

        rmssd, sdnn = last_rmssd, last_sdnn
            
        # --- SEÑALES CONTINUAS (Extraídas del df_unido a 1Hz) ---
        v_hr = df_m_idx.loc[window_start:t, 'HR'].values
        v_eda = df_m_idx.loc[window_start:t, 'EDA'].values
        v_temp = df_m_idx.loc[window_start:t, 'TEMP'].values
        v_acc = df_m_idx.loc[window_start:t, 'Movimiento'].values

        # --- VALOR INSTANTÁNEO EN t (sin ventana) ---
        hr_inst = df_m_idx.loc[t, 'HR']
        eda_inst = df_m_idx.loc[t, 'EDA']
        temp_inst = df_m_idx.loc[t, 'TEMP']
        acc_inst = df_m_idx.loc[t, 'Movimiento']
        
        # --- COLAPSO DE ETIQUETA DE REFERENCIA ---
        v_etq = df_m_idx.loc[window_start:t, 'Etiqueta'].values
        if len(v_etq) > 0:
            etiqueta_ventana = int(df_m_idx.loc[t, 'Etiqueta'])
            ventana_pura = bool(np.all(v_etq == v_etq[0]))
        else:
            etiqueta_ventana = 0
            ventana_pura = False
        
        features_list.append({
            'Tiempo': t,
            'HR_inst': hr_inst,
            'HR_roll60s_mean': np.mean(v_hr) if len(v_hr) > 0 else np.nan,
            'HRV_RMSSD': rmssd,
            'HRV_SDNN': sdnn,
            'ACC_inst': acc_inst,
            'ACC_roll60s_mean': np.mean(v_acc) if len(v_acc) > 0 else np.nan,
            'ACC_roll60s_max': np.max(v_acc) if len(v_acc) > 0 else np.nan,
            'EDA_inst': eda_inst,
            'EDA_roll60s_mean': np.mean(v_eda) if len(v_eda) > 0 else np.nan,
            'EDA_roll60s_std': np.std(v_eda) if len(v_eda) > 0 else np.nan,
            'TEMP_inst': temp_inst,
            'TEMP_roll60s_mean': np.mean(v_temp) if len(v_temp) > 0 else np.nan,
            'Etiqueta': etiqueta_ventana,
            'ventana_pura': ventana_pura
        })
        
    df_features = pd.DataFrame(features_list).ffill().bfill()
    
    # Mapeo final de estados emocionales
    mapa_emociones = {0: 'Transicion', 1: 'Relajado', 2: 'Estresado', 3: 'Diversion', 4: 'Meditacion', 5: 'Otros', 6: 'Otros', 7: 'Otros'}
    df_features['Estado_Emocional'] = df_features['Etiqueta'].map(mapa_emociones)
    
    return df_features

### 0.2.3. procesar_fisiologia_sujeto(id_sujeto, config, sampling_mode='1hz')
Esta función orquesta, para un único sujeto, todo el recorrido desde los ficheros crudos del Empatica E4 hasta el dataset de características listo para modelado. Se estructura en siete checkpoints secuenciales:

**Checkpoint 1 — Verificación de rutas y archivos esenciales**
Antes de procesar nada, se comprueba que existan físicamente los ficheros necesarios del sujeto (ACC, HR, EDA, TEMP e IBI del Empatica E4, más el `.pkl` de etiquetas de WESAD). Si falta alguno de los archivos críticos, la función se detiene devolviendo `None` en lugar de fallar a mitad de pipeline. IBI y TAGS no se consideran críticos, ya que su ausencia se gestiona más adelante como caso particular.

**Checkpoint 2 — Sincronización de relojes base**
Cada sensor del E4 registra su propio timestamp Unix de inicio en la primera línea del CSV. Este checkpoint extrae ese instante inicial para ACC y HR por separado, ya que no está garantizado que ambos arranquen en el mismo segundo exacto. Ese instante servirá como origen de tiempo (t=0) para reconstruir el eje temporal real de cada señal.

**Checkpoint 3 — Remuestreo de señales continuas a 1Hz**
ACC (32Hz), EDA (4Hz) y TEMP (4Hz) se agregan por promedio a resolución de 1 segundo, mientras que HR ya viene nativamente a 1Hz. Para ACC, se calcula primero la magnitud del vector de aceleración (√(x²+y²+z²)) antes de promediar, de modo que el resultado representa intensidad de movimiento y no una componente direccional arbitraria.

**Checkpoint 4 — Procesamiento de eventos discretos (IBI y tags)**
El IBI (intervalo entre latidos) se mantiene deliberadamente en su resolución nativa guiada por eventos, sin forzar un remuestreo a 1Hz que destruiría la distancia real latido a latido — información necesaria para el cálculo posterior de RMSSD/SDNN. Los tags (pulsaciones de botón del sujeto durante el experimento) se cargan de forma tolerante: si el fichero no existe o está vacío, se genera un DataFrame vacío en lugar de interrumpir el procesamiento del sujeto.

**Checkpoint 5 — Fusión de la matriz base a 1Hz**
HR, ACC, EDA y TEMP se fusionan mediante `inner join` sobre la columna de tiempo, garantizando que la tabla resultante tenga exactamente una fila por segundo sin duplicados. Los tags se incorporan aparte mediante `left join`, rellenando con 0 los segundos sin pulsación de botón, ya que su ausencia es informativa (no hay evento) y no debe generar pérdida de filas.

**Checkpoint 6 — Sincronización fina de etiquetas vía curva térmica**
Los timestamps del CSV del E4 y del `.pkl` de etiquetas de WESAD no comparten el mismo origen temporal de forma fiable, por lo que no pueden alinearse por reloj directamente. Este checkpoint resuelve el desfase deslizando la serie de temperatura del `.pkl` sobre la del CSV y localizando el desplazamiento (offset) que minimiza el error absoluto medio entre ambas curvas térmicas. Ese offset óptimo se usa como ancla real de sincronización para recortar el CSV y asignarle las etiquetas de estado correspondientes.

**Checkpoint 7 — Extracción de características en ventanas móviles**
Con la matriz de 1Hz ya etiquetada y sincronizada, se invoca `extract_physiological_features()` (sección 0.2.2), que calcula los estadísticos por ventana móvil de 1 minuto y colapsa la etiqueta de estado. Este es el punto de entrega del pipeline por sujeto: el DataFrame devuelto aquí es el que posteriormente se concatena entre sujetos en el pipeline global.

In [7]:
def procesar_fisiologia_sujeto(id_sujeto, config, sampling_mode='1hz'):
    """
    Carga, unifica, alinea térmicamente y extrae características en ventanas
    para un sujeto específico en el modo de frecuencia indicado ('1hz' o '1min').
    """
    print(f"\n🚀 === PROCESANDO SUJETO: {id_sujeto} [Modo: {sampling_mode}] ===")
    
    # --- CHECKPOINT 1: Extracción de rutas dinámicas ---
    ruta_dir_e4 = Path(config["mapa_sujetos"][id_sujeto]["e4_dir"])
    ruta_pkl = Path(config["mapa_sujetos"][id_sujeto]["pkl_file"])
    
    archivos_esenciales = {
        "ACC": ruta_dir_e4 / "ACC.csv",
        "HR": ruta_dir_e4 / "HR.csv",
        "EDA": ruta_dir_e4 / "EDA.csv",
        "TEMP": ruta_dir_e4 / "TEMP.csv",
        "IBI": ruta_dir_e4 / "IBI.csv",
        "TAGS": ruta_dir_e4 / "tags.csv",
        "PKL": ruta_pkl
    }
    # Se valida que los archivos críticos existan antes de perder tiempo computando
    for nombre, ruta in archivos_esenciales.items():
        if nombre not in ["IBI", "TAGS"] and not ruta.exists():
            print(f"   ❌ ERROR CRÍTICO: No se encuentra el archivo de {nombre} en la ruta: {ruta}")
            return None
    print("   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.")
    
    # --- CHECKPOINT 2: Definición de los Relojes Base ---
    with open(archivos_esenciales["ACC"], 'r') as f:
        unix_base_acc = float(f.readline().strip().split(',')[0])
    inicio_reloj = unix_a_hora_real(unix_base_acc)

    with open(archivos_esenciales["HR"], 'r') as f:
        unix_base_hr = float(f.readline().strip().split(',')[0])
    inicio_reloj_hr = unix_a_hora_real(unix_base_hr)
    print(f"   [✓] Checkpoint 2: Relojes base sincronizados (ACC: {inicio_reloj.time()} | HR: {inicio_reloj_hr.time()}).")
    
    # --- CHECKPOINT 3: Carga y Reducción Fisiológica a 1 Hz ---
    f_acc = config["frecuencias"]["ACC"]
    f_eda = config["frecuencias"]["EDA"]
    f_temp = config["frecuencias"]["TEMP"]
    
    # 3.1 HR (1 Hz)
    df_hr = pd.read_csv(archivos_esenciales["HR"], skiprows=2, header=None, names=['HR'])
    df_hr['Tiempo'] = [inicio_reloj_hr + datetime.timedelta(seconds=i) for i in range(len(df_hr))]
    
    # 3.2 ACC (32 Hz -> 1 Hz)
    df_acc_raw = pd.read_csv(archivos_esenciales["ACC"], skiprows=2, header=None, names=['X', 'Y', 'Z'])
    df_acc_raw['Magnitud'] = np.sqrt(df_acc_raw['X']**2 + df_acc_raw['Y']**2 + df_acc_raw['Z']**2)
    acc_por_segundo = df_acc_raw.groupby(np.arange(len(df_acc_raw)) // f_acc)['Magnitud'].mean().values
    df_acc = pd.DataFrame({'Movimiento': acc_por_segundo, 'Tiempo': [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(acc_por_segundo))]})
    
    # 3.3 EDA (4 Hz -> 1 Hz)
    df_eda_raw = pd.read_csv(archivos_esenciales["EDA"], skiprows=2, header=None, names=['EDA'])
    eda_por_segundo = df_eda_raw.groupby(np.arange(len(df_eda_raw)) // f_eda)['EDA'].mean().values
    df_eda = pd.DataFrame({'EDA': eda_por_segundo, 'Tiempo': [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(eda_por_segundo))]})
    
    # 3.4 TEMP (4 Hz -> 1 Hz)
    df_temp_raw = pd.read_csv(archivos_esenciales["TEMP"], skiprows=2, header=None, names=['TEMP'])
    temp_por_segundo = df_temp_raw.groupby(np.arange(len(df_temp_raw)) // f_temp)['TEMP'].mean().values
    df_temp = pd.DataFrame({'TEMP': temp_por_segundo, 'Tiempo': [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(temp_por_segundo))]})
    
    print(f"   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.")
    
    # --- CHECKPOINT 4: Procesamiento de IBI CRUDO (Alta resolución basada en eventos) ---
    try:
        df_ibi_raw = pd.read_csv(archivos_esenciales["IBI"], skiprows=1, names=['Segundos_Transcurridos', 'Intervalo'])
        # Se conservan los decimales exactos de los segundos para no destruir la distancia real latido a latido
        df_ibi_raw['Tiempo'] = [inicio_reloj + datetime.timedelta(seconds=float(seg)) for seg in df_ibi_raw['Segundos_Transcurridos']]
        df_ibi_raw['Intervalo_ms'] = df_ibi_raw['Intervalo'] * 1000
        
        # IMPORTANTE: NO se redondea a '1s' ni se agrupa por segundo. Se mantiene la serie pura guiada por eventos.
        df_ibi = df_ibi_raw[['Tiempo', 'Intervalo_ms']].copy().sort_values('Tiempo')
    except (pd.errors.EmptyDataError, pd.errors.ParserError, KeyError, ValueError) as e:
        print(f"   ⚠️ Aviso: Estructura de IBI no estándar o vacía ({type(e).__name__}: {e}). Se generará dataframe vacío.")
        df_ibi = pd.DataFrame(columns=['Tiempo', 'Intervalo_ms'])
        
    # 4.2 Tags (Pulsaciones de botón)
    if archivos_esenciales["TAGS"].exists() and archivos_esenciales["TAGS"].stat().st_size > 0:
        df_tags_raw = pd.read_csv(archivos_esenciales["TAGS"], header=None, names=['Unix'])
        df_tags = pd.DataFrame()
        df_tags['Tiempo'] = pd.to_datetime(df_tags_raw['Unix'], unit='s').dt.round('1s')
        df_tags['Boton_Pulsado'] = 1

    else:
        df_tags = pd.DataFrame(columns=['Tiempo', 'Boton_Pulsado'])
        
    print("   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.")
    
    # --- CHECKPOINT 5: Fusión de la Matriz Base a 1 Hz  ---
    # Al NO meter el IBI aquí, se garantiza que df_unido sea estrictamente de 1 fila por segundo (sin duplicaciones artificiales)
    df_unido = pd.merge(df_hr[['HR', 'Tiempo']], df_acc[['Movimiento', 'Tiempo']], on='Tiempo', how='inner')
    df_unido = pd.merge(df_unido, df_eda[['EDA', 'Tiempo']], on='Tiempo', how='inner')
    df_unido = pd.merge(df_unido, df_temp[['TEMP', 'Tiempo']], on='Tiempo', how='inner')
    df_unido = pd.merge(df_unido, df_tags[['Boton_Pulsado', 'Tiempo']], on='Tiempo', how='left')
    df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)
    print(f"   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: {df_unido.shape[0]}")
    
    # --- CHECKPOINT 6: Sincronización Temporal Real vía Curva Térmica ---
    with open(archivos_esenciales["PKL"], 'rb') as f:
        data_pkl = pickle.load(f, encoding='latin1')
        
    etiquetas_700hz = data_pkl['label']
    temp_wrist_4hz = data_pkl['signal']['wrist']['TEMP'].flatten()
    f_pkl = config["frecuencias"]["pkl"]
    
    # Reducción de etiquetas a 1 Hz por Moda
    etiquetas_por_segundo = [int(pd.Series(etiquetas_700hz[i:i+f_pkl]).mode()[0]) for i in range(0, len(etiquetas_700hz), f_pkl) if len(etiquetas_700hz[i:i+f_pkl]) > 0]
     # Reducción de temperatura pkl de muñeca a 1 Hz por Media
    temp_wrist_por_segundo = [float(temp_wrist_4hz[i:i+4].mean()) for i in range(0, len(temp_wrist_4hz), 4) if len(temp_wrist_4hz[i:i+4]) > 0]
    
    # Se convierten los perfiles térmicos a vectores de Numpy para un cálculo ultrarrápido
    arr_csv = df_unido['TEMP'].values
    arr_pkl = np.array(temp_wrist_por_segundo)
    n_csv, n_pkl = len(arr_csv), len(arr_pkl)
    
    best_offset = 0
    min_error = float('inf')

    # Se desliza el pkl sobre el CSV para encontrar el encaje perfecto de la curva de temperatura
    if n_csv >= n_pkl:
        for offset in range(0, n_csv - n_pkl + 1):
            sub_csv = arr_csv[offset : offset + n_pkl]
            error = np.mean(np.abs(sub_csv - arr_pkl))
            if error < min_error:
                min_error = error
                best_offset = offset

        # Se recorta el dataset para seleccionar estrictamente con la ventana del experimento real
        df_unido = df_unido.iloc[best_offset : best_offset + n_pkl].copy().reset_index(drop=True)
        df_unido['Etiqueta'] = etiquetas_por_segundo
    else:
        # Caso de seguridad: si por algún fallo el CSV fuera más corto que el pkl
        print(f"   ⚠️ Alerta: El CSV es más corto que el PKL. Forzando alineación.")
        df_unido['Etiqueta'] = etiquetas_por_segundo[:n_csv]
        min_error = 0.0

    # Se mapean los estados emocionales sobre el recorte perfecto
    mapa_emociones = {0: 'Transicion', 1: 'Relajado', 2: 'Estresado', 3: 'Diversion', 4: 'Meditacion', 5: 'Otros', 6: 'Otros', 7: 'Otros'}
    df_unido['Estado_Emocional'] = df_unido['Etiqueta'].map(mapa_emociones)
    
        
    print(f"   [✓] Checkpoint 6: Sincronización por curva térmica completada. ")
    print(f"       -> Segundo de inicio (Offset): {best_offset}s | Error de ajuste: {min_error:.4f}°C")
    
    # --- CHECKPOINT 7: EXTRACCIÓN EN VENTANAS MÓVILES (integración IBI) ---
    # Se envia la matriz limpia etiquetada de 1Hz y la serie independiente de IBI cruda
    df_features = extract_physiological_features(df_unido, df_ibi, sampling_mode=sampling_mode)
    
    return df_features

### 0.2.4. Normalización por baseline
Esta función normaliza las variables fisiológicas mediante un Z-score referenciado al propio estado de reposo (baseline) de cada sujeto, en lugar de usar la media y desviación típica global de la muestra. Para cada sujeto se calculan media y desviación típica exclusivamente sobre las filas etiquetadas como condición basal (`label_baseline`), y esos dos estadísticos —específicos de cada individuo— se aplican para tipificar la totalidad de sus registros, incluidas las fases de estrés, diversión o meditación.

Esta decisión es fisiológicamente necesaria antes de entrenar cualquier clasificador entre sujetos: cada persona tiene un HR o una conductancia de la piel basal distintos, y sin este anclaje individual el modelo aprendería diferencias inter-sujeto en lugar de la respuesta de estrés que realmente interesa detectar. Se añade una constante de estabilidad (`1e-8`) al denominador para evitar divisiones por cero en el caso límite de una desviación típica basal nula.

In [8]:
import pandas as pd
import numpy as np

# Definimos las columnas fisiológicas a normalizar
columnas_a_normalizar = None

def aplicar_normalizacion_por_baseline(df_global, col_label='label', label_baseline=1, columnas_a_normalizar=None):
    """
    Normaliza las características fisiológicas del dataframe maestro calculando
    la media y std ÚNICAMENTE en el estado de reposo (baseline) de cada sujeto.
    
    Parameters:
    -----------
    df_global : pd.DataFrame
        Dataset concatenado con todos los sujetos.
    col_label : str
        Nombre de la columna que contiene la etiqueta del estado (ej. 1=baseline).
    label_baseline : int o list
        Valor(es) de la etiqueta correspondiente al estado relajado/basal.
    """
    print(f"\n⚖️ Normalizando variables fisiológicas usando la condición BASAL (label={label_baseline})...")

    if columnas_a_normalizar is None: raise ValueError("Debe especificarse una lista de columnas a normalizar.")

    # 1. Filtrar solo los datos basales
    df_baseline = df_global[df_global[col_label] == label_baseline]
    
    # 2. Calcular media y std basales agrupando por 'ID_sujeto'
    stats_basales = df_baseline.groupby('ID_sujeto')[columnas_a_normalizar].agg(['mean', 'std'])
    
    # 3. Aplicar la normalización a todo el dataset global
    for col in columnas_a_normalizar:
        # Extraemos media y std del baseline para cada sujeto mediante mapeo
        mean_map = stats_basales[(col, 'mean')]
        std_map = stats_basales[(col, 'std')]
        
        # Mapeamos los valores basales a las filas correspondientes según ID_sujeto
        sujeto_mean = df_global['ID_sujeto'].map(mean_map)
        sujeto_std = df_global['ID_sujeto'].map(std_map)
        
        # Z-score basado estrictamente en el estado de reposo
        df_global[f'{col}_norm'] = (df_global[col] - sujeto_mean) / (sujeto_std + 1e-8)
        
    return df_global



### 0.2.5. Cálculo del timestamp
Función auxiliar que convierte un timestamp Unix a datetime, fijando siempre UTC como referencia y descartando la información de zona horaria en el resultado. Esto garantiza que la reconstrucción del eje temporal sea reproducible independientemente de la zona horaria configurada en la máquina donde se ejecute el notebook.

In [9]:
def unix_a_hora_real(unix_timestamp):
    """
    Convierte un timestamp Unix a datetime "naive" (sin tzinfo) usando
    SIEMPRE UTC como referencia, para que el resultado no dependa de
    la configuración de zona horaria del sistema operativo donde se ejecute.
    """
    return datetime.datetime.fromtimestamp(unix_timestamp, tz=datetime.timezone.utc).replace(tzinfo=None)

# 1. Pipeline para generar los datos a 1Hz

## 1.1. Prueba para un sujeto individual (S7)
Antes de automatizar el pipeline para los 15 sujetos, se desarrolla y valida cada paso sobre un único sujeto de referencia (S7). Este enfoque permite inspeccionar visualmente los datos crudos, detectar problemas de formato o sincronización, y confirmar que la lógica es correcta antes de generalizarla al resto de la muestra.

### 1.1.1. Configuración de rutas e inspección preliminar
Se selecciona S7 como sujeto de prueba y se cargan sus rutas específicas desde el mapa de configuración externo. A continuación se inspeccionan las primeras líneas de cada CSV (ACC, EDA, HR, IBI, TEMP) para verificar el formato de los ficheros del Empatica E4 antes de procesarlos.

In [10]:
import json
import pickle
from pathlib import Path
import pandas as pd
import datetime

# ======================================================================================================
# 🚀 SELECCIÓN DE SUJETO Y CARGA DEL MAPA DE RUTAS EXTERNO
# ======================================================================================================

# Única variable que se cambiará en el código (o que se meterá dentro de un bucle 'for')
sujeto_a_procesar = "S7"



# Se verifica si el sujeto solicitado existe en el mapa de rutas
if sujeto_a_procesar not in config["mapa_sujetos"]:
    raise KeyError(f"❌ El sujeto '{sujeto_a_procesar}' no está registrado en el archivo config_rutas.json")

# Se extraen las rutas específicas del sujeto elegido directamente del mapa del JSON
ruta_data = Path(config["mapa_sujetos"][sujeto_a_procesar]["e4_dir"])
ruta_pkl = Path(config["mapa_sujetos"][sujeto_a_procesar]["pkl_file"])


print(f"🌍 Procesando de forma dinámica al sujeto: {sujeto_a_procesar}")
print(f"📍 Carpeta E4 mapeada: {ruta_data}")
print(f"📦 Archivo .pkl mapeado: {ruta_pkl}\n")

🌍 Procesando de forma dinámica al sujeto: S7
📍 Carpeta E4 mapeada: ..\data\raw\WESAD\S7\S7_E4_Data
📦 Archivo .pkl mapeado: ..\data\raw\WESAD\S7\S7.pkl



In [11]:
# ==========================================
# 🔍 INSPECCIÓN DE LAS PRIMERAS LÍNEAS DE TODOS LOS CSV
# ==========================================
archivos = [
    "ACC.csv",
    "EDA.csv",
    "HR.csv",
    "IBI.csv",
    "TEMP.csv"
]
for nombre_doc in archivos:
    ruta_file = ruta_data / nombre_doc
    
    if ruta_file.exists():
        with open(ruta_file, 'r') as f:
            primeras_filas = [f.readline().strip() for _ in range(5)]
        
        print(f"✅ Archivo encontrado: {nombre_doc}")
        print("-" * 50)
        for i, fila in enumerate(primeras_filas):
            print(f"  Línea {i+1}: {fila}")
        print("-" * 50 + "\n")
    else:
        print(f"❌ Error: No se encontró el archivo {nombre_doc} en la ruta.\n")

✅ Archivo encontrado: ACC.csv
--------------------------------------------------
  Línea 1: 1499339464.000000, 1499339464.000000, 1499339464.000000
  Línea 2: 32.000000, 32.000000, 32.000000
  Línea 3: 7,5,61
  Línea 4: 7,5,60
  Línea 5: 4,5,59
--------------------------------------------------

✅ Archivo encontrado: EDA.csv
--------------------------------------------------
  Línea 1: 1499339464.000000
  Línea 2: 4.000000
  Línea 3: 0.000000
  Línea 4: 0.863495
  Línea 5: 1.206600
--------------------------------------------------

✅ Archivo encontrado: HR.csv
--------------------------------------------------
  Línea 1: 1499339474.000000
  Línea 2: 1.000000
  Línea 3: 85.00
  Línea 4: 73.00
  Línea 5: 76.33
--------------------------------------------------

✅ Archivo encontrado: IBI.csv
--------------------------------------------------
  Línea 1: 1499339464.000000, IBI
  Línea 2: 11.344269,0.921917
  Línea 3: 12.313064,0.968794
  Línea 4: 13.297484,0.984420
  Línea 5: 14.266278,0.9

La inspección confirma que los cinco ficheros del Empatica E4 están disponibles para S7 y siguen el formato estándar: las dos primeras líneas codifican el timestamp Unix de inicio y la frecuencia de muestreo, seguidas de los datos crudos. Se observa además que HR.csv arranca en `1499339474`, diez segundos después que ACC, EDA e IBI (`1499339464`) — confirmando empíricamente la necesidad, ya anticipada en el Checkpoint 2 de 0.2.3, de tratar el reloj base de HR de forma independiente al del resto de sensores.

### 1.1.2. Procesamiento de los archivos .csv disponibles para el sujeto

En los datos descargados se dispone del archivo .csv independiente de las diferentes características medidas con el reloj de muñeca: 
-  HR
-  ACC
-  BVP
-  EDA
-  IBI
-  TEMP
-  tags  
   
Se analizarán los datos disponibles en cada uno de estos .csv para validar su contenido antes de fusionarlos en un único dataset, con la excepción de BVP.   
Aunque el dataset multimodal WESAD incluye la señal cruda de Pulso de Volumen Sanguíneo (bvp.csv), esta es descartada del pipeline de ingeniería de características por motivos de redundancia y eficiencia computacional. Dado que los archivos de Frecuencia Cardíaca (HR.csv) e Intervalos Inter-latido (IBI.csv) proporcionados por el dispositivo ya constituyen variables derivadas y validadas a partir del procesamiento algorítmico del BVP, la inclusión de la señal cruda no aportaría información adicional para el cálculo de la HRV y aumentaría innecesariamente la dimensionalidad y el coste computacional del modelo.

In [12]:
# ======================================================================================================
# 🔄 PROCESAMIENTO PASO A PASO 
# ======================================================================================================

print("=" * 60)
print("📊 EXTRACCIÓN DE METADATOS Y TIEMPOS DE INICIO")
print("=" * 60)

# --- 1. FRECUENCIA CARDÍACA (HR.csv) ---
ruta_hr = ruta_data / "HR.csv"
if ruta_hr.exists():
    with open(ruta_hr, 'r') as f:
        # HR solo tiene un número en la primera línea
        inicio_hr_unix = float(f.readline().strip().split(',')[0]) 
        frecuencia_hr = float(f.readline().strip().split(',')[0])
    
    hora_hr = unix_a_hora_real(inicio_hr_unix)
    print(f"💓 [HR]  Inició: {hora_hr} | Frecuencia original: {frecuencia_hr} Hz")


# --- 2. ACELERÓMETRO (ACC.csv) ---
ruta_acc = ruta_data / "ACC.csv"
if ruta_acc.exists():
    with open(ruta_acc, 'r') as f:
        # ACC tiene tres números repetidos, se usa .split(',')[0] para coger solo el primero
        inicio_acc_unix = float(f.readline().strip().split(',')[0])
        frecuencia_acc = float(f.readline().strip().split(',')[0])
        
    hora_acc = unix_a_hora_real(inicio_acc_unix)
    print(f"🏃‍♂️ [ACC] Inició: {hora_acc} | Frecuencia original: {frecuencia_acc} Hz")


# --- 3. PULSO DE VOLUMEN SANGUÍNEO (BVP.csv) ---
ruta_bvp = ruta_data / "BVP.csv"
if ruta_bvp.exists():
    with open(ruta_bvp, 'r') as f:
        inicio_bvp_unix = float(f.readline().strip().split(',')[0])
        frecuencia_bvp = float(f.readline().strip().split(',')[0])
        
    hora_bvp = unix_a_hora_real(inicio_bvp_unix)
    print(f"🩸 [BVP] Inició: {hora_bvp} | Frecuencia original: {frecuencia_bvp} Hz")


# --- 4. RESPUESTA GALVÁNICA / SUDOR (EDA.csv) ---
ruta_eda = ruta_data / "EDA.csv"
if ruta_eda.exists():
    with open(ruta_eda, 'r') as f:
        inicio_eda_unix = float(f.readline().strip().split(',')[0])
        frecuencia_eda = float(f.readline().strip().split(',')[0])
        
    hora_eda = unix_a_hora_real(inicio_eda_unix)
    print(f"💧 [EDA] Inició: {hora_eda} | Frecuencia original: {frecuencia_eda} Hz")


# --- 5. INTERVALO ENTRE LATIDOS (IBI.csv) ---
ruta_ibi = ruta_data / "IBI.csv"
if ruta_ibi.exists():
    with open(ruta_ibi, 'r') as f:
        # El IBI suele tener texto en la línea 1 ("Time, Duration") en lugar de un Unix timestamp directo
        inicio_ibi_unix = float(f.readline().strip().split(',')[0])
    hora_ibi = unix_a_hora_real(inicio_ibi_unix)    
        
    print(f"⏱️ [IBI] Inició: {hora_ibi} (Segun archivo info este no va a frecuencia fija, sino que capta intervalos entre latidos)")


# --- 6. TEMPERATURA (TEMP.csv) ---
ruta_temp = ruta_data / "TEMP.csv"
if ruta_temp.exists():
    with open(ruta_temp, 'r') as f:
        inicio_temp_unix = float(f.readline().strip().split(',')[0])
        frecuencia_temp = float(f.readline().strip().split(',')[0])
        
    hora_temp = unix_a_hora_real(inicio_temp_unix)
    print(f"🌡️  [TEMP] Inició: {hora_temp} | Frecuencia original: {frecuencia_temp} Hz")

# --- 7. ETIQUETAS (tags.csv) ---
ruta_tags = ruta_data / "tags.csv"
if ruta_tags.exists():
    with open(ruta_tags, 'r') as f:
        # Las etiquetas suelen tener texto en la primera línea ("Time, Label") en lugar de un Unix timestamp directo
        inicio_tags_unix = float(f.readline().strip().split(',')[0])
    hora_tags = unix_a_hora_real(inicio_tags_unix)    
        
    print(f"🏷️ [tags] Inició: {hora_tags} (Segun archivo info este no va a frecuencia fija, sino que capta eventos de etiqueta)")

📊 EXTRACCIÓN DE METADATOS Y TIEMPOS DE INICIO
💓 [HR]  Inició: 2017-07-06 11:11:14 | Frecuencia original: 1.0 Hz
🏃‍♂️ [ACC] Inició: 2017-07-06 11:11:04 | Frecuencia original: 32.0 Hz
🩸 [BVP] Inició: 2017-07-06 11:11:04 | Frecuencia original: 64.0 Hz
💧 [EDA] Inició: 2017-07-06 11:11:04 | Frecuencia original: 4.0 Hz
⏱️ [IBI] Inició: 2017-07-06 11:11:04 (Segun archivo info este no va a frecuencia fija, sino que capta intervalos entre latidos)
🌡️  [TEMP] Inició: 2017-07-06 11:11:04 | Frecuencia original: 4.0 Hz
🏷️ [tags] Inició: 2017-07-06 12:59:17.860000 (Segun archivo info este no va a frecuencia fija, sino que capta eventos de etiqueta)


De aquí se observa que de los datos del acelerometro el tiempo de inicio es de 10 segundos antes que el resto de sensores, por lo que hay que sincronizar por tiempo los datos de los distintos csv.

In [13]:
import pandas as pd
import numpy as np
import datetime

# ======================================================================================================
#  DEFINICIÓN DEL RELOJ BASE DINÁMICO (A partir del Acelerómetro)
# ======================================================================================================


with open(ruta_acc, 'r') as f:
    # Se extrae el Timestamp Unix real de la primera línea de la base de forma dinámica
    unix_base_acc = float(f.readline().strip().split(',')[0])

# Este es el objeto datetime de referencia local para TODO el sujeto actual
inicio_reloj = unix_a_hora_real(unix_base_acc)

with open(ruta_hr, 'r') as f:
    # Se extrae el Timestamp Unix real de la primera línea de la base de forma dinámica
    unix_base_hr = float(f.readline().strip().split(',')[0])

# Este es el objeto datetime de referencia local para TODO el sujeto actual
inicio_reloj_hr = unix_a_hora_real(unix_base_hr)


print(f"⏰ Tiempo Base del Reloj: {inicio_reloj}")
print(f"⏰ Tiempo Base de HR:    {inicio_reloj_hr}\n")

# ======================================================================================================
# 1. CARGA DE FRECUENCIA CARDÍACA (HR) - 1 Hz (1 dato por segundo)
# ======================================================================================================
df_hr = pd.read_csv(ruta_hr, skiprows=2, header=None, names=['HR'])
# Al ir a 1 Hz, se suma un segundo por cada fila
df_hr['Tiempo'] = [inicio_reloj_hr + datetime.timedelta(seconds=i) for i in range(len(df_hr))]

print(f"💓 HR cargado: {df_hr.shape[0]} filas.")


# ======================================================================================================
# 2. CARGA DE ACELERÓMETRO (ACC) - 32 Hz (32 datos por segundo) -> Reducimos a 1 Hz con la media
# ======================================================================================================
df_acc_raw = pd.read_csv(ruta_acc, skiprows=2, header=None, names=['X', 'Y', 'Z'])
df_acc_raw['Magnitud'] = np.sqrt(df_acc_raw['X']**2 + df_acc_raw['Y']**2 + df_acc_raw['Z']**2)

# Se agrupan de 32 en 32 filas para calcular la media de cada segundo
acc_por_segundo = df_acc_raw.groupby(np.arange(len(df_acc_raw)) // 32)['Magnitud'].mean().values

df_acc = pd.DataFrame({'Movimiento': acc_por_segundo})
# Al reducirlo a segundos, se suma un segundo por cada posición desde 'inicio_reloj'
df_acc['Tiempo'] = [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(df_acc))]

print(f"🏃‍♂️ ACC transformado a segundos: {df_acc.shape[0]} filas.")


# ======================================================================================================
# 3. CARGA DE SUDORACIÓN (EDA) - 4 Hz (4 datos por segundo) -> Reducimos a 1 Hz con la media
# ======================================================================================================
df_eda_raw = pd.read_csv(ruta_eda, skiprows=2, header=None, names=['EDA'])

# Se agrupan de 4 en 4 filas para calcular la media de cada segundo
eda_por_segundo = df_eda_raw.groupby(np.arange(len(df_eda_raw)) // 4)['EDA'].mean().values

df_eda = pd.DataFrame({'EDA': eda_por_segundo})
# Al reducirlo a segundos, se suma un segundo por cada posición desde 'inicio_reloj'
df_eda['Tiempo'] = [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(df_eda))]

print(f"💧 EDA transformado a segundos: {df_eda.shape[0]} filas.")

# ======================================================================================================
# 4. CARGA DE TEMPERATURA (TEMP) - 4 Hz (4 datos por segundo) -> Reducimos a 1 Hz con la media
# ======================================================================================================
df_temp_raw = pd.read_csv(ruta_temp, skiprows=2, header=None, names=['TEMP'])

# Se agrupan de 4 en 4 filas para calcular la media de cada segundo
temp_por_segundo = df_temp_raw.groupby(np.arange(len(df_temp_raw)) // 4)['TEMP'].mean().values

df_temp = pd.DataFrame({'TEMP': temp_por_segundo})
# Al reducirlo a segundos, se suma un segundo por cada posición desde 'inicio_reloj'
df_temp['Tiempo'] = [inicio_reloj + datetime.timedelta(seconds=i) for i in range(len(df_temp))]

print(f"🌡️ TEMP transformado a segundos: {df_temp.shape[0]} filas.")

# ======================================================================================================
# 4.5. NUEVO: CARGA DE MARCAS DE BOTÓN (tags.csv) -> No lleva frecuencia fija, son eventos únicos
# ======================================================================================================
df_tags_raw = pd.read_csv(ruta_data / "tags.csv", header=None, names=['Unix'])

# Se convierten cada timestamp del archivo tags directamente en una fecha y hora real legible
df_tags = pd.DataFrame()
df_tags['Tiempo'] = pd.to_datetime(df_tags_raw['Unix'], unit='s').dt.round('1s')
df_tags['Boton_Pulsado'] = 1


print(f"🏷️ tags procesado: {df_tags.shape[0]} pulsaciones registradas.")


# ======================================================================================================
# 5. INSPECCIÓN VISUAL DE TUS DATAFRAMES INDEPENDIENTES
# ======================================================================================================
print("\nMuestra del df_hr:")
print(df_hr.head(3))

print("\nMuestra del df_acc (ya reducido por segundos):")
print(df_acc.head(3))

print("\nMuestra del df_eda (ya reducido por segundos):")
print(df_eda.head(3))

print("\nMuestra del df_temp (ya reducido por segundos):")
print(df_temp.head(3))

print("\nMuestra del df_tags (eventos de pulsación):")
print(df_tags.head(3))

⏰ Tiempo Base del Reloj: 2017-07-06 11:11:04
⏰ Tiempo Base de HR:    2017-07-06 11:11:14

💓 HR cargado: 6485 filas.
🏃‍♂️ ACC transformado a segundos: 6495 filas.
💧 EDA transformado a segundos: 6494 filas.
🌡️ TEMP transformado a segundos: 6494 filas.
🏷️ tags procesado: 1 pulsaciones registradas.

Muestra del df_hr:
      HR              Tiempo
0  85.00 2017-07-06 11:11:14
1  73.00 2017-07-06 11:11:15
2  76.33 2017-07-06 11:11:16

Muestra del df_acc (ya reducido por segundos):
   Movimiento              Tiempo
0   62.743969 2017-07-06 11:11:04
1   63.193002 2017-07-06 11:11:05
2   62.964051 2017-07-06 11:11:06

Muestra del df_eda (ya reducido por segundos):
        EDA              Tiempo
0  0.928567 2017-07-06 11:11:04
1  2.863364 2017-07-06 11:11:05
2  4.638675 2017-07-06 11:11:06

Muestra del df_temp (ya reducido por segundos):
     TEMP              Tiempo
0  382.21 2017-07-06 11:11:04
1   33.21 2017-07-06 11:11:05
2   33.21 2017-07-06 11:11:06

Muestra del df_tags (eventos de pulsac

El dataset IBI contiene datos por evento, no muestreados, por lo que su tratamiento se realizará de forma distinta. 

In [14]:
# ======================================================================================================
# 6. CARGA Y ASIGNACIÓN DE TIEMPO REAL AL ARCHIVO IBI 
# ======================================================================================================

# Se carga el IBI saltando la primera línea que es la cabecera Unix
df_ibi = pd.read_csv(ruta_data / "IBI.csv", skiprows=1, names=['Segundos_Transcurridos', 'Intervalo'])

# Se convierten los segundos transcurridos en fechas y horas reales en base al inicio del reloj (13:11:04)
inicio_reloj = unix_a_hora_real(1499339464.0)

df_ibi['Tiempo_Real'] = [inicio_reloj + datetime.timedelta(seconds=float(seg)) for seg in df_ibi['Segundos_Transcurridos']]

# Se redondea la hora al segundo exacto más cercano para poder emparejarlo después
df_ibi['Tiempo'] = df_ibi['Tiempo_Real'].dt.round('1s')

# Se pasa el intervalo de segundos a milisegundos (multiplicando por 1000), que es el estándar médico para HRV
df_ibi['Intervalo_ms'] = df_ibi['Intervalo'] * 1000

print(f"⏱️ Archivo IBI cargado con éxito: {df_ibi.shape[0]} latidos detectados en total.")
print("\nPrimeras 5 filas del procesamiento de IBI con tiempos reales:")
print(df_ibi[['Segundos_Transcurridos', 'Intervalo_ms', 'Tiempo']].head(20))

⏱️ Archivo IBI cargado con éxito: 2948 latidos detectados en total.

Primeras 5 filas del procesamiento de IBI con tiempos reales:
    Segundos_Transcurridos  Intervalo_ms              Tiempo
0                11.344269       921.917 2017-07-06 11:11:15
1                12.313064       968.794 2017-07-06 11:11:16
2                13.297484       984.420 2017-07-06 11:11:17
3                14.266278       968.794 2017-07-06 11:11:18
4                15.110067       843.789 2017-07-06 11:11:19
5                15.969481       859.414 2017-07-06 11:11:20
6                16.844521       875.040 2017-07-06 11:11:21
7                17.672684       828.163 2017-07-06 11:11:22
8                18.500847       828.163 2017-07-06 11:11:23
9                19.422764       921.917 2017-07-06 11:11:23
10               20.407184       984.420 2017-07-06 11:11:24
11               21.329101       921.917 2017-07-06 11:11:25
12               22.266644       937.543 2017-07-06 11:11:26
13             

**Conclusión y Justificación Metodológica: Procesamiento del IBI**

**1. El Problema Técnico (Colisiones en el Grid de Tiempo)**

La señal de intervalos inter-latido (IBI) es de naturaleza *event-driven* (guiada por eventos), por lo que carece de una frecuencia de muestreo fija. Fisiológicamente, es habitual que ocurran dos latidos dentro de un mismo segundo del reloj.

Si se realizara una fusión directa (*merge*) de este IBI crudo sobre el grid regular de 1Hz del resto de bioseñales (HR, EDA, TEMP, ACC), Pandas duplicaría las filas enteras de esos segundos para no perder ningún latido, rompiendo la continuidad uniforme del tiempo y distorsionando el cálculo de cualquier ventana móvil posterior.

**2. La Solución Diseñada (Extracción por Ventanas Móviles)**

Para mantener un paso de tiempo constante — requisito estricto para modelos de Machine Learning como Random Forest —, se descarta la fusión del IBI instantáneo. En su lugar, se aplica un pipeline de extracción de características en ventanas móviles de 60 segundos, que procesa los latidos crudos de forma aislada para calcular biomarcadores consolidados de Variabilidad de la Frecuencia Cardíaca (HRV): **RMSSD** y **SDNN**. Una vez transformados los eventos en estas series continuas, la unificación con el dataset maestro se realiza de forma segura y sin duplicados.

**3. Justificación y Alineación con los Objetivos del TFM**

Esta arquitectura de datos es fundamental para el proyecto por dos motivos:
- **Integridad matemática:** sincroniza todas las bioseñales en una matriz limpia, evitando la propagación de errores en el cálculo de `StressLoad(t)`.
- **Simetría con el "mundo real":** los wearables comerciales de consumo no exportan el latido crudo segundo a segundo, sino métricas resumen (como el RMSSD) en bloques de tiempo. Procesar el baseline de WESAD con esta misma estrategia garantiza que el modelo sea metodológicamente transferible al escenario de baja resolución temporal (1 minuto) que constituye el núcleo del TFM.

### 1.1.3. Unificación de datos en un único dataframe

Se unen las cuatro señales continuas (HR, ACC, EDA, TEMP) mediante `inner join` sobre la columna de tiempo, garantizando una fila por segundo sin duplicados. Adicionalmente, se incorporan las marcas de botón (`tags`) mediante `left join`, rellenando con 0 los segundos sin pulsación.

In [15]:
# ======================================================================================================
# 7. UNIFICACIÓN DE SEÑALES USANDO LA COLUMNA DE TIEMPO REAL (DISEÑO FINAL RESISTENTE)
# ======================================================================================================

print("🔄 Iniciando fusión limpia desde los dataframes base...")

# 1. Se asegura la fusión estricta de las 4 señales continuas esenciales
df_unido = pd.merge(df_hr[['HR', 'Tiempo']], df_acc[['Movimiento', 'Tiempo']], on='Tiempo', how='inner')
df_unido = pd.merge(df_unido, df_eda[['EDA', 'Tiempo']], on='Tiempo', how='inner')
df_unido = pd.merge(df_unido, df_temp[['TEMP', 'Tiempo']], on='Tiempo', how='inner')

# 3. Se unen las marcas de botón del tag
df_unido = pd.merge(df_unido, df_tags[['Boton_Pulsado', 'Tiempo']], on='Tiempo', how='left')
df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)

# 4. Se reordenan estéticamente las columnas para el informe 
columnas_finales = ['Tiempo', 'HR',  'Movimiento', 'EDA', 'TEMP']
df_unido = df_unido[columnas_finales]

print(f"\n✅ ¡Ecosistema Empatica E4 integrado perfectamente!")
print(f"Dimensiones de la tabla unificada: {df_unido.shape}")

# ======================================================================================================
# 🔍 CONTROL DE CALIDAD VISUAL
# ======================================================================================================
print("\nPrimeras filas del dataset maestro:")
print(df_unido.head(12))

print("\nÚltimas filas del dataset maestro:")
print(df_unido.tail(5))

🔄 Iniciando fusión limpia desde los dataframes base...

✅ ¡Ecosistema Empatica E4 integrado perfectamente!
Dimensiones de la tabla unificada: (6484, 5)

Primeras filas del dataset maestro:
                Tiempo     HR  Movimiento       EDA   TEMP
0  2017-07-06 11:11:14  85.00   62.919800  5.126800  33.29
1  2017-07-06 11:11:15  73.00   62.988052  5.145670  33.29
2  2017-07-06 11:11:16  76.33   62.999825  5.145989  33.29
3  2017-07-06 11:11:17  71.75   62.989058  5.153345  33.29
4  2017-07-06 11:11:18  69.20   62.846330  5.176371  33.31
5  2017-07-06 11:11:19  69.83   62.896341  5.182767  33.33
6  2017-07-06 11:11:20  70.43   62.977564  5.188524  33.34
7  2017-07-06 11:11:21  70.88   62.832338  5.181808  33.37
8  2017-07-06 11:11:22  71.22   62.840859  5.177011  33.34
9  2017-07-06 11:11:23  71.50   62.879690  5.180848  33.37
10 2017-07-06 11:11:24  71.73   62.871530  5.175412  33.39
11 2017-07-06 11:11:25  71.33   62.921073  5.175412  33.39

Últimas filas del dataset maestro:
        

La fusión mediante `inner join` sobre las cuatro señales continuas produce una tabla estrictamente regular de 6.484 filas (una por segundo), sin pérdida de continuidad temporal, cubriendo el registro completo de S7 desde las 11:11:14 hasta las 12:59:17. La inspección de cabecera y cola confirma que no hay saltos ni duplicados introducidos por la fusión.

### 1.1.4. Procesado del archivo .pkl para añadir las etiquetas de estado al dataset

Con los pasos anteriores ya se dispone del dataset completo a 1Hz para el sujeto seleccionado, pero es necesario aún incluir las etiquetas de estado. Estas se obtienen del archivo `.pkl`, que a diferencia de los CSV del Empatica E4 no contiene ningún timestamp: sus señales y etiquetas son arrays secuenciales indexados por posición de muestra, sin referencia a un reloj absoluto. Por tanto, no es posible alinear ambas fuentes por marca temporal; en su lugar, se utiliza como ancla la señal de temperatura de muñeca, presente en ambas fuentes, para identificar mediante ajuste de curvas el desplazamiento a partir del cual los datos del `.pkl` se corresponden con los del CSV, y se emplean las etiquetas a partir de dicho punto.

In [16]:
# ======================================================================================================
# CARGA Y ALINEACIÓN DE ETIQUETAS USANDO ÚNICAMENTE DATOS DE LA MUÑECA (WRIST)
# ======================================================================================================
print(f"⏳ Leyendo archivo .pkl para extraer etiquetas y señales de la muñeca...")

with open(ruta_pkl, 'rb') as f:
    data_pkl = pickle.load(f, encoding='latin1')

# 1. Se extraen las etiquetas (700 Hz) y la temperatura de la MUÑECA (wrist) del pkl (4 Hz)
etiquetas_700hz = data_pkl['label']
temp_wrist_4hz = data_pkl['signal']['wrist']['TEMP'].flatten()

print(f"📏 Registros de etiquetas a 700 Hz: {len(etiquetas_700hz)}")
print(f"🌡️ Registros de temperatura de muñeca en el .pkl a 4 Hz: {len(temp_wrist_4hz)}")

# 2. Se reducen las etiquetas a 1 Hz (agrupando de 700 en 700 por moda)
etiquetas_por_segundo = []
for i in range(0, len(etiquetas_700hz), 700):
    bloque_etq = etiquetas_700hz[i:i+700]
    if len(bloque_etq) > 0:
        etiquetas_por_segundo.append(int(pd.Series(bloque_etq).mode()[0]))

# 3. Se reducen la temperatura de muñeca del pkl a 1 Hz (agrupando de 4 en 4 por media)
temp_wrist_por_segundo = []
for i in range(0, len(temp_wrist_4hz), 4):
    bloque_tmp = temp_wrist_4hz[i:i+4]
    if len(bloque_tmp) > 0:
        temp_wrist_por_segundo.append(float(bloque_tmp.mean()))

print(f"🏷️ Vector de etiquetas a 1 Hz: {len(etiquetas_por_segundo)} segundos.")
print(f"🌡️ Vector de TEMP (wrist) del .pkl a 1 Hz: {len(temp_wrist_por_segundo)} segundos.")


⏳ Leyendo archivo .pkl para extraer etiquetas y señales de la muñeca...
📏 Registros de etiquetas a 700 Hz: 3666600
🌡️ Registros de temperatura de muñeca en el .pkl a 4 Hz: 20952
🏷️ Vector de etiquetas a 1 Hz: 5238 segundos.
🌡️ Vector de TEMP (wrist) del .pkl a 1 Hz: 5238 segundos.


La carga confirma que el `.pkl` contiene 3.666.600 muestras de etiqueta a 700Hz y 20.952 muestras de temperatura de muñeca a 4Hz, ambas reducidas a 5.238 segundos tras el colapso a 1Hz. La alineación por ventana deslizante localiza el desplazamiento óptimo en 1.169 segundos, con un error absoluto medio de 0.0000°C entre ambas curvas térmicas — un ajuste esencialmente perfecto, como cabía esperar al tratarse del mismo sensor físico bajo dos codificaciones de tiempo distintas. El dataset de S7 queda así recortado y etiquetado en 5.238 filas, listas para la extracción de características.

In [17]:
# ======================================================================================================
# SINCRO REAL: ALINEACIÓN TEMPORAL POR VENTANA DESLIZANTE (MAE DE TEMPERATURA)
# ======================================================================================================
print("\n🔄 Sincronizando etiquetas mediante Ventana Deslizante (Minimización de MAE)...")

import numpy as np

# Convertimos las series de temperatura a arrays de numpy para un cálculo ultra rápido
temp_csv = df_unido['TEMP'].values
temp_pkl = np.array(temp_wrist_por_segundo)

n_csv = len(temp_csv)
n_pkl = len(temp_pkl)

# Inicializamos las variables para buscar el mejor acoplamiento temporal
best_offset = 0
min_mae = float('inf')

# Deslizamos el bloque del .pkl (segundo a segundo) a lo largo del CSV completo de la pulsera
for offset in range(0, n_csv - n_pkl + 1):
    sub_csv = temp_csv[offset : offset + n_pkl]
    mae = np.mean(np.abs(sub_csv - temp_pkl))
    
    if mae < min_mae:
        min_mae = mae
        best_offset = offset

print(f"📍 Sincronización temporal completada con éxito:")
print(f"   - Mejor desplazamiento (best_offset) encontrado: {best_offset} segundos.")
print(f"   - Error Absoluto Medio (MAE) de la curva térmica: {min_mae:.4f} °C")

# ------------------------------------------------------------------------------------------------------
# RECORTE Y ASIGNACIÓN SECUENCIAL CRONOLÓGICA
# ------------------------------------------------------------------------------------------------------
# Recortamos df_unido para quedarnos EXACTAMENTE con el fragmento de tiempo que duró el .pkl
df_unido = df_unido.iloc[best_offset : best_offset + n_pkl].copy().reset_index(drop=True)

# Como ya están perfectamente alineados segundo a segundo, inyectamos las etiquetas directamente
df_unido['Etiqueta'] = etiquetas_por_segundo

# Se mapean los nombres de los estados emocionales
mapa_emociones = {0: 'Transicion', 1: 'Relajado', 2: 'Estresado', 3: 'Diversion', 4: 'Meditacion', 5: 'Otros', 6: 'Otros', 7: 'Otros'}
df_unido['Estado_Emocional'] = df_unido['Etiqueta'].map(mapa_emociones)

print(f"\n🎉 ¡DATASET MAESTRO DE MUÑECA PARA EL SUJETO {sujeto_a_procesar} COMPLETADO!")
print(f"Dimensiones finales: {df_unido.shape} (Recorte perfecto sincronizado cronológicamente)")

print("\nDistribución del tiempo por Estado Emocional (en segundos):")
print(df_unido['Estado_Emocional'].value_counts())


🔄 Sincronizando etiquetas mediante Ventana Deslizante (Minimización de MAE)...
📍 Sincronización temporal completada con éxito:
   - Mejor desplazamiento (best_offset) encontrado: 1169 segundos.
   - Error Absoluto Medio (MAE) de la curva térmica: 0.0000 °C

🎉 ¡DATASET MAESTRO DE MUÑECA PARA EL SUJETO S7 COMPLETADO!
Dimensiones finales: (5238, 7) (Recorte perfecto sincronizado cronológicamente)

Distribución del tiempo por Estado Emocional (en segundos):
Estado_Emocional
Transicion    2103
Relajado      1186
Meditacion     790
Estresado      640
Diversion      372
Otros          147
Name: count, dtype: int64


La sincronización por ventana deslizante localiza el desplazamiento óptimo en 1.169 segundos, con un error absoluto medio de la curva térmica de 0.0000°C — un ajuste prácticamente perfecto, coherente con el hecho de que ambas series de temperatura (CSV y `.pkl`) provienen del mismo sensor de muñeca, solo que referenciadas a relojes distintos. Tras el recorte, el dataset de S7 queda alineado en 5.238 filas etiquetadas, distribuidas en seis estados emocionales (Transición, Relajado, Meditación, Estresado, Diversión y Otros), confirmando que el desajuste horario detectado en la exploración preliminar queda resuelto mediante este anclaje por temperatura.

### 1.1.5. Inclusión de los datos obtenidos a partir de la información de IBI

Además de las señales continuas ya integradas, se incorporan al dataset los dos biomarcadores de variabilidad cardíaca calculados a partir del IBI crudo:

-  SDNN (Standard Deviation of NN intervals): Es la desviación estándar de todos los intervalos inter-latido (los llamados intervalos NN o RR) en la ventana de tiempo. En el código    se     calcula con np.std(v_ibi, ddof=1).  
    El SDNN mide la variabilidad total del corazón en ese bloque de tiempo. Refleja tanto la actividad del sistema nervioso simpático (el acelerador/estrés) como del parasimpático (el freno/relajación).

    Un SDNN alto indica un corazón flexible y con buena capacidad de adaptación.

    Un SDNN bajo sostenido suele ser un indicador de fatiga, estrés crónico o una baja resiliencia del organismo (carga alostática).  

-  RMSSD (Root Mean Square of Successive Differences): Es la raíz cuadrada de la media de las diferencias sucesivas al cuadrado entre latidos adyacentes.
  A diferencia del SDNN, el RMSSD mide exclusivamente las variaciones latido a latido a muy corto plazo. Fisiológicamente, este cambio tan rápido está mediado de forma casi pura por el sistema nervioso parasimpático (a través del nervio vago).

    El RMSSD es el indicador por excelencia de la capacidad de recuperación y relajación del cuerpo.

    Cuando el sujeto se estresa, el freno parasimpático se retira, el corazón se vuelve más rígido (los latidos se vuelven muy monótonos y rítmicos) y el RMSSD cae en picado.

**Inferencia del Estrés**: Al combinar ambos en un modelo ML, el modelo puede contrastar la variabilidad total (SDNN) contra la actividad de relajación parasimpática (RMSSD). Si el RMSSD baja de forma drástica mientras el HR (ritmo cardíaco) sube, el modelo detectará con mucha precisión que el sujeto está entrando en el estado Estresado

In [18]:
print(df_unido['Tiempo'].dtype, df_unido['Tiempo'].min(), df_unido['Tiempo'].max())
print(df_ibi['Tiempo'].dtype, df_ibi['Tiempo'].min(), df_ibi['Tiempo'].max())
print(df_unido.shape, df_ibi.shape)

datetime64[ns] 2017-07-06 11:30:43 2017-07-06 12:58:00
datetime64[ns] 2017-07-06 11:11:15 2017-07-06 12:58:57
(5238, 7) (2948, 5)


In [19]:
# Se lanza la ingenieria de caracteristicas
# Ahora se genera el dataset listo para Machine Learning
df_unido_final = extract_physiological_features(df_unido, df_ibi, sampling_mode='1hz', hrv_stride_s=30)

# Control de calidad
print(df_unido_final['Estado_Emocional'].value_counts())

🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...
Estado_Emocional
Transicion    2043
Relajado      1186
Meditacion     790
Estresado      640
Diversion      372
Otros          147
Name: count, dtype: int64


Tras integrar el IBI mediante extracción por ventanas móviles, el dataset final de S7 queda con 5.178 filas (2.043 Transición, 1.186 Relajado, 790 Meditación, 640 Estresado, 372 Diversión, 147 Otros) — 60 filas menos que el dataset de entrada (5.238), correspondientes al primer minuto de la grabación, descartado porque la ventana móvil de 60 segundos necesita ese margen antes de producir su primer valor válido. La redistribución de clases respecto al conteo previo a las ventanas (celda 36) es coherente con el filtro de regularización temporal esperado: la moda por ventana absorbe transiciones espurias de corta duración.

## 1.2. Pipeline global para todos los sujetos
Validada la lógica de procesamiento sobre el sujeto de prueba S7, se extiende el pipeline mediante `procesar_fisiologia_sujeto()` a los 15 sujetos disponibles del dataset WESAD. Cada sujeto se procesa de forma independiente, y sus resultados se concatenan en una única matriz maestra, sobre la que finalmente se aplica la normalización por baseline (0.2.4) para eliminar la variabilidad fisiológica inter-sujeto.

In [20]:
import json
import pickle
import datetime
import numpy as np
import pandas as pd
from pathlib import Path

# ======================================================================================================
# 🧭 1. CARGA DEL MAPA DE CONFIGURACIÓN GLOBAL
# ======================================================================================================


# Se extrae la lista de todos los sujetos declarados en el JSON
lista_sujetos = list(config["mapa_sujetos"].keys())

print(f"📋 Archivo de configuración cargado correctamente.")
print(f"Frecuencias base -> ACC: {config['frecuencias']['ACC']}Hz, EDA: {config['frecuencias']['EDA']}Hz, TEMP: {config['frecuencias']['TEMP']}Hz, PKL: {config['frecuencias']['pkl']}Hz")
print(f"Sujetos detectados listos para procesar ({len(lista_sujetos)}): {lista_sujetos}")

📋 Archivo de configuración cargado correctamente.
Frecuencias base -> ACC: 32Hz, EDA: 4Hz, TEMP: 4Hz, PKL: 700Hz
Sujetos detectados listos para procesar (15): ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']


In [21]:
# Hacemos un test con el sujeto S7 que ya conocemos bien
df_test = procesar_fisiologia_sujeto("S7", config, "1hz")

print("\n🔍 Inspección rápida del DataFrame devuelto:")
print(df_test.head(3))

print(df_test['Estado_Emocional'].value_counts())


🚀 === PROCESANDO SUJETO: S7 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 11:11:04 | HR: 11:11:14).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 6484
   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1169s | Error de ajuste: 0.0000°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🔍 Inspección rápida del DataFrame devuelto:
               Tiempo  HR_inst  HR_roll60s_mean  HRV_RMSSD   HRV_SDNN  \
0 2017-07-06 11:31:43    78.55        82.213934  59.176222  78.362325   
1 2017-07-06 11:31:44    78.12        82.235082  59.176222  78.362325   
2 2017-07-06 11:31:45    77.70        82.238689  59.176222  78.362325   

    ACC_inst  ACC_roll60s_mean  ACC_roll60s_max  EDA_inst  EDA_roll6

Los resultados coinciden con los del pipeline anterior para el sujeto S7, asi que se considera que el pipeline automático funciona correctamente.   
Por lo tanto se pasa a ejecutar el pipeline para todos los sujetos y para las frecuencias 1hz y 1min

La construcción de `df_unido` en `procesar_fisiologia_sujeto()` asume que ACC, EDA y TEMP comparten el mismo timestamp de inicio (verificado manualmente solo para S7 en la exploración preliminar), reutilizando el reloj de ACC para las tres señales, y que HR arranca con un desfase fijo. Antes de generalizar esta asunción a los 15 sujetos, se verifica leyendo directamente la cabecera de cada CSV crudo.

In [22]:
verificacion_relojes = []
for sujeto in lista_sujetos:
    ruta_dir = Path(config["mapa_sujetos"][sujeto]["e4_dir"])
    with open(ruta_dir / "ACC.csv") as f: t_acc = float(f.readline().split(',')[0])
    with open(ruta_dir / "EDA.csv") as f: t_eda = float(f.readline().split(',')[0])
    with open(ruta_dir / "TEMP.csv") as f: t_temp = float(f.readline().split(',')[0])
    with open(ruta_dir / "HR.csv") as f: t_hr = float(f.readline().split(',')[0])
    verificacion_relojes.append({
        'ID_sujeto': sujeto,
        'ACC_eq_EDA': t_acc == t_eda,
        'ACC_eq_TEMP': t_acc == t_temp,
        'desfase_HR_s': t_hr - t_acc
    })
pd.DataFrame(verificacion_relojes)

,ID_sujeto,ACC_eq_EDA,ACC_eq_TEMP,desfase_HR_s
0,S2,True,True,10.0
1,S3,True,True,10.0
2,S4,True,True,10.0
3,S5,True,True,10.0
4,S6,True,True,10.0
5,S7,True,True,10.0
6,S8,True,True,10.0
7,S9,True,True,10.0
8,S10,True,True,10.0
9,S11,True,True,10.0


In [23]:
# Lista oficial de sujetos disponibles en el dataset proxy WESAD
# lista_sujetos = ["S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10", "S11", "S13", "S14", "S15", "S16", "S17"]

# Ejecutamos de forma automática el pipeline para ambas resoluciones requeridas en el TFM
for modo_frecuencia in ['1hz']:
    print(f"\n=======================================================================")
    print(f"🧱 INICIANDO GENERACIÓN DEL DATASET MAESTRO EN FRECUENCIA: [{modo_frecuencia}]")
    print(f"=======================================================================")
    
    datasets_sujetos = []
    
    # Paso 2: Extender el pipeline a todos los sujetos
    for sujeto in lista_sujetos:
        try:
            df_sujeto = procesar_fisiologia_sujeto(sujeto, config, sampling_mode=modo_frecuencia)
            if df_sujeto is not None and not df_sujeto.empty:
                # 🚀 LÍNEA CORREGIDA: Inyectamos el ID del sujeto antes de acumularlo
                df_sujeto['ID_sujeto'] = sujeto
                datasets_sujetos.append(df_sujeto)
        except Exception as e:
            print(f"❌ Error crítico procesando al sujeto {sujeto}: {str(e)}")
            
    # Concatenamos todos los sujetos procesados en una única gran matriz de características
    df_maestro_global = pd.concat(datasets_sujetos, ignore_index=True)
    
    # Paso 3: Normalizar los resultados mediante Baselines Personalizados
    # Fisiológicamente, cada individuo tiene medias y desviaciones distintas de HR o EDA.
    # Aplicamos Z-Score agrupando por 'ID_sujeto' para remover la variabilidad inter-sujeto.
    columnas_a_normalizar = [
    'HR_inst', 'HR_roll60s_mean',
    'HRV_RMSSD', 'HRV_SDNN',
    'ACC_inst', 'ACC_roll60s_mean', 'ACC_roll60s_max',
    'EDA_inst', 'EDA_roll60s_mean', 'EDA_roll60s_std',
    'TEMP_inst', 'TEMP_roll60s_mean'
]
    
    print(f"\n⚖️ Normalizando variables fisiológicas mediante baselines por sujeto (Z-Score)...")

    # --- USO DENTRO DE TU PIPELINE ---

    df_maestro_global = aplicar_normalizacion_por_baseline(df_maestro_global, col_label='Etiqueta', label_baseline=1, columnas_a_normalizar=columnas_a_normalizar)
    
    
    # Paso 4 y 5: Guardar el archivo de resultados definitivo
    #ruta_salida_final = f"WESAD_features_maestro_{modo_frecuencia}.csv"
    #df_maestro_global.to_csv(ruta_salida_final, index=False)
    
    #print(f"\n💾 [PROCESO COMPLETADO] Dataset guardado con éxito.")
    #print(f"   --> Archivo: {ruta_salida_final}")
    print(f"   --> Dimensiones finales de la matriz para ML: {df_maestro_global.shape}")


🧱 INICIANDO GENERACIÓN DEL DATASET MAESTRO EN FRECUENCIA: [1hz]

🚀 === PROCESANDO SUJETO: S2 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 07:15:25 | HR: 07:15:35).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7864


C:\Users\miner\AppData\Local\Temp\ipykernel_134104\1118443529.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)


   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1535s | Error de ajuste: 0.0000°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S3 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 11:08:48 | HR: 11:08:58).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7712


C:\Users\miner\AppData\Local\Temp\ipykernel_134104\1118443529.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)


   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1080s | Error de ajuste: 0.0066°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S4 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 08:33:40 | HR: 08:33:50).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7988
   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1465s | Error de ajuste: 0.0063°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S5 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 12:22:13 | HR: 12:22:23).
   [✓] Chec

In [24]:
# ======================================================================================================
# 💾 ALMACENAMIENTO PERMANENTE DEL DATASET DE PROCESAMIENTO
# ======================================================================================================
from pathlib import Path

# Extraemos las rutas directamente del archivo de configuración JSON
ruta_salida_parquet_1Hz = Path(config["rutas_procesamiento"]["ruta_salida_parquet_1Hz"])
ruta_salida_csv = Path(config["rutas_procesamiento"]["ruta_salida_csv_1Hz"])

# Nos aseguramos de que la carpeta 'processed' exista en tu disco
ruta_salida_parquet_1Hz.parent.mkdir(parents=True, exist_ok=True)

print(f"💾 Guardando Dataset Maestro unificado con {df_maestro_global.shape[1]} columnas...")

# Guardamos en Parquet (Altamente recomendado: mantiene tipos de datos, fechas y comprime el tamaño)
# EL TRUCO: Forzamos el motor 'fastparquet' para saltarnos el bug de pyarrow
df_maestro_global.to_parquet(ruta_salida_parquet_1Hz, engine='fastparquet', index=False)
print(f"   [✓] Guardado con éxito en formato Parquet: {ruta_salida_parquet_1Hz}")

# Opcional: Descomenta la línea de abajo si necesitas inspeccionarlo visualmente en Excel
# df_maestro_global.to_csv(ruta_salida_csv, index=False, encoding='utf-8')
# print(f"   [✓] Guardado con éxito en formato CSV: {ruta_salida_csv}")

print("\n🎉 ¡ENHORABUENA! ¡PROCESO DE INGENIERÍA DE DATOS FINALIZADO! 🎉")
print(f"Tu matriz global definitiva tiene unas dimensiones perfectas de: {df_maestro_global.shape}")
print("El dataset está 100% limpio, sincronizado dinámicamente y listo para la fase de modelado de IA.")

print("Columnas actuales en df_maestro_global:")
for i, col in enumerate(df_maestro_global.columns):
    print(f"{i+1}. {col}")

💾 Guardando Dataset Maestro unificado con 29 columnas...
   [✓] Guardado con éxito en formato Parquet: ..\data\processed\WESAD_dataset_maestro_1Hz.parquet

🎉 ¡ENHORABUENA! ¡PROCESO DE INGENIERÍA DE DATOS FINALIZADO! 🎉
Tu matriz global definitiva tiene unas dimensiones perfectas de: (85069, 29)
El dataset está 100% limpio, sincronizado dinámicamente y listo para la fase de modelado de IA.
Columnas actuales en df_maestro_global:
1. Tiempo
2. HR_inst
3. HR_roll60s_mean
4. HRV_RMSSD
5. HRV_SDNN
6. ACC_inst
7. ACC_roll60s_mean
8. ACC_roll60s_max
9. EDA_inst
10. EDA_roll60s_mean
11. EDA_roll60s_std
12. TEMP_inst
13. TEMP_roll60s_mean
14. Etiqueta
15. ventana_pura
16. Estado_Emocional
17. ID_sujeto
18. HR_inst_norm
19. HR_roll60s_mean_norm
20. HRV_RMSSD_norm
21. HRV_SDNN_norm
22. ACC_inst_norm
23. ACC_roll60s_mean_norm
24. ACC_roll60s_max_norm
25. EDA_inst_norm
26. EDA_roll60s_mean_norm
27. EDA_roll60s_std_norm
28. TEMP_inst_norm
29. TEMP_roll60s_mean_norm


El pipeline automático reproduce exactamente los resultados obtenidos en la prueba manual sobre S7, confirmando su corrección antes de extenderlo al conjunto completo de sujetos. Tras procesar los 15 sujetos, el dataset maestro resultante contiene 85.069 filas y 29 columnas, incluyendo tanto las variables crudas como sus versiones normalizadas por Z-score (baseline-referenced) descritas en 0.2.4. El resultado se almacena en formato Parquet, preservando tipos de datos y fechas de forma más eficiente que un CSV plano.

# 2. Pipeline para generar los datos a 1/60 Hz

## 2.1. Justificación de la agregación a resolución de 1 minuto

El dataset WESAD proporciona señales fisiológicas a alta resolución (hasta 700Hz para las etiquetas, 32Hz para el acelerómetro), muy por encima de lo que ofrecen los wearables comerciales de consumo. Estos dispositivos —incluido el Xiaomi Smart Watch S4 utilizado en el estudio n-of-1 de este TFM— suelen reportar métricas resumen a intervalos del orden del minuto, no la señal cruda. Por ello, se construye una segunda versión del dataset agregada a 1/60Hz, que replica la granularidad real disponible en el escenario de aplicación final del sistema.

## 2.2. Pipeline de agregación y generación de nuevas características

Partiendo del dataset a 1Hz ya construido en el bloque 1, se agrupan las filas de cada sujeto en bloques de 60 segundos, calculando la media de cada señal por minuto y colapsando la etiqueta de estado por moda. Para compensar la pérdida de detalle inherente a esta reducción de resolución, se añaden además ventanas de contexto temporal macroscópico (medias y desviaciones móviles a 3, 5 y 10 minutos) y variables de lag (t-1, t-2, t-3), que aportan al modelo información sobre la tendencia reciente más allá del valor puntual de cada minuto. El resultado se normaliza de nuevo mediante Z-score por baseline (0.2.4) y se almacena como el dataset definitivo de trabajo a 1/60Hz.

In [25]:

lista_sujetos = list(config["mapa_sujetos"].keys())
# lista_sujetos = ["S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10", "S11", "S13", "S14", "S15", "S16", "S17"]


for modo_frecuencia in ['1hz']:
    print(f"\n=======================================================================")
    print(f"🧱 INICIANDO GENERACIÓN DEL DATASET MAESTRO EN FRECUENCIA: [1/60hz]")
    print(f"=======================================================================")
    
    datasets_sujetos_ml = []
    
    # --------------------------------------------------------------------------------------------------
    # PASO 1: PROCESAMIENTO GENERAL, REDUCCIÓN A 1 MINUTO Y RENOMBRADO (Por sujeto)
    # --------------------------------------------------------------------------------------------------
    for sujeto in lista_sujetos:
        try:
            # Se prepara el dataset en modo 1Hz para posteriormente a lo largo del bucle realizar 
            # agregaciones por minuto y renombrado de columnas
            df_sujeto = procesar_fisiologia_sujeto(sujeto, config, sampling_mode=modo_frecuencia)
            if df_sujeto is not None and not df_sujeto.empty:
                df_sujeto['ID_sujeto'] = sujeto
                df_sujeto = df_sujeto.reset_index(drop=True)
                # Se crea una columna 'minuto_id' que indica a qué minuto pertenece cada fila (0, 1, 2, ...) para hacer las agregaciones por minuto
                df_sujeto['minuto_id'] = (df_sujeto.index // 60) # de 0 a 59 es minuto 0, de 60 a 119 es minuto 1, etc.
                
                # Configuración adaptada a las columnas que entrega procesar_fisiologia_sujeto() para el modo 1Hz
                agregaciones = {
                    'HR_inst': ['mean'],
                    'HRV_RMSSD': ['mean'],
                    'HRV_SDNN': ['mean'],
                    'ACC_inst': ['mean'],
                    'EDA_inst': ['mean'],
                    'TEMP_inst': ['mean'],
                    'Estado_Emocional': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0],
                    'Etiqueta': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0],
                    'ventana_pura': 'all'
                }
                # Se agrupa por 'ID_sujeto' y 'minuto_id' para obtener las estadísticas por minuto
                df_sujeto_ml = df_sujeto.groupby(['ID_sujeto', 'minuto_id']).agg(agregaciones).reset_index()
                df_sujeto_ml.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df_sujeto_ml.columns.values]
                
                # Renombrado inteligente
                diccionario_renombrado = {
                    'HR_inst_mean': 'HR_mean',
                    'HRV_RMSSD_mean': 'HRV_RMSSD_mean',
                    'HRV_SDNN_mean': 'HRV_SDNN_mean',
                    'ACC_inst_mean': 'ACC_mean',
                    'EDA_inst_mean': 'EDA_mean',
                    'TEMP_inst_mean': 'TEMP_mean',
                    'Estado_Emocional_<lambda>': 'Estado_Emocional',
                    'Etiqueta_<lambda>': 'Etiqueta',
                    'ventana_pura_all': 'ventana_pura',
                    'ID_sujeto_': 'ID_sujeto',
                    'minuto_id_': 'minuto_id'
                }
                df_sujeto_ml = df_sujeto_ml.rename(columns=diccionario_renombrado)
                datasets_sujetos_ml.append(df_sujeto_ml)
                
        except Exception as e:
            print(f"❌ Error crítico procesando al sujeto {sujeto}: {str(e)}")
            
    # Concatenamos de forma segura para tener la gran matriz base
    df_maestro_global = pd.concat(datasets_sujetos_ml, ignore_index=True)
    
    # --------------------------------------------------------------------------------------------------
    # PASO 2: AÑADIR CONTEXTO TEMPORAL MACROSCÓPICO (Ventanas de 5 y 10 minutos)
    # --------------------------------------------------------------------------------------------------
    print("\n🔄 Añadiendo contexto temporal macroscópico (Ventanas de 3, 5 y 10 minutos)...")
    df_maestro_global = df_maestro_global.sort_values(by=['ID_sujeto', 'minuto_id']).reset_index(drop=True)
    
    # Columnas base adaptadas para calcularles las tendencias macro
    columnas_base_rolling = ['HR_mean', 'HRV_RMSSD_mean', 'EDA_mean', 'TEMP_mean', 'ACC_mean']

    # Ventana de 3 minutos
    for col in columnas_base_rolling:
        if col in df_maestro_global.columns:
            df_maestro_global[f'{col}_roll_mean_3m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=3, min_periods=1).mean()
            )
            df_maestro_global[f'{col}_roll_std_3m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=3, min_periods=1).std().fillna(0)
            )
    
    # Ventana de 5 minutos
    for col in columnas_base_rolling:
        if col in df_maestro_global.columns:
            df_maestro_global[f'{col}_roll_mean_5m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=5, min_periods=1).mean()
            )
            df_maestro_global[f'{col}_roll_std_5m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=5, min_periods=1).std().fillna(0)
            )
            
    # Ventana de 10 minutos
    for col in columnas_base_rolling:
        if col in df_maestro_global.columns:
            df_maestro_global[f'{col}_roll_mean_10m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=10, min_periods=1).mean()
            )
            df_maestro_global[f'{col}_roll_std_10m'] = df_maestro_global.groupby('ID_sujeto')[col].transform(
                lambda x: x.rolling(window=10, min_periods=1).std().fillna(0)
            )

    # --------------------------------------------------------------------------------------------------
    # PASO 3: NORMALIZACIÓN Z-SCORE GLOBAL MEDIANTE BASELINES POR SUJETO
    # --------------------------------------------------------------------------------------------------
    print(f"\n⚖️ Normalizando variables fisiológicas mediante baselines por sujeto (Z-Score)...")

    # Declaramos la lista completa de las 36 características que necesitan Z-Score por sujeto
    columnas_a_normalizar = [
        # --- 1. CARACTERÍSTICAS BASE DE 1 MINUTO (6 columnas) ---
        'HR_mean', 
        'HRV_RMSSD_mean', 'HRV_SDNN_mean',
        'ACC_mean', 
        'EDA_mean', 
        'TEMP_mean', 

        # --- 2. TENDENCIAS MACRO DE 3 MINUTOS (10 columnas) ---
        'HR_mean_roll_mean_3m',         'HR_mean_roll_std_3m',
        'HRV_RMSSD_mean_roll_mean_3m',  'HRV_RMSSD_mean_roll_std_3m',
        'EDA_mean_roll_mean_3m',        'EDA_mean_roll_std_3m',
        'TEMP_mean_roll_mean_3m',       'TEMP_mean_roll_std_3m',
        'ACC_mean_roll_mean_3m',        'ACC_mean_roll_std_3m',

        # --- 3. TENDENCIAS MACRO DE 5 MINUTOS (10 columnas) ---
        'HR_mean_roll_mean_5m',         'HR_mean_roll_std_5m',
        'HRV_RMSSD_mean_roll_mean_5m',  'HRV_RMSSD_mean_roll_std_5m',
        'EDA_mean_roll_mean_5m',        'EDA_mean_roll_std_5m',
        'TEMP_mean_roll_mean_5m',       'TEMP_mean_roll_std_5m',
        'ACC_mean_roll_mean_5m',        'ACC_mean_roll_std_5m',

        # --- 3. TENDENCIAS MACRO DE 10 MINUTOS (10 columnas) ---
        'HR_mean_roll_mean_10m',        'HR_mean_roll_std_10m',
        'HRV_RMSSD_mean_roll_mean_10m', 'HRV_RMSSD_mean_roll_std_10m',
        'EDA_mean_roll_mean_10m',       'EDA_mean_roll_std_10m',
        'TEMP_mean_roll_mean_10m',      'TEMP_mean_roll_std_10m',
        'ACC_mean_roll_mean_10m',       'ACC_mean_roll_std_10m'
    ]


    # --- USO DENTRO DE TU PIPELINE ---
    # df_maestro_global = pd.concat(datasets_sujetos, ignore_index=True)
    df_maestro_global = aplicar_normalizacion_por_baseline(df_maestro_global, col_label='Etiqueta', label_baseline=1, columnas_a_normalizar=columnas_a_normalizar)
    

    # --------------------------------------------------------------------------------------------------
    # PASO 4: CÁLCULO DE LAGS TEMPORALES AVANZADOS (Sobre columnas ya normalizadas)
    # --------------------------------------------------------------------------------------------------
    print("🔄 Generando lags temporales (t-1, t-2, t-3) de inercia cardíaca y motora...")
    # Ajustamos 'Movimiento_mean_norm' a su nombre real actual: 'ACC_mean_norm'
    columnas_lags = ['HR_mean_norm', 'HRV_RMSSD_mean_norm', 'EDA_mean_norm', 'TEMP_mean_norm']
    
    for col in columnas_lags:
        if col in df_maestro_global.columns:
            # Lag 1 minuto (t-1)
            df_maestro_global[f'{col}_lag_1m_norm'] = df_maestro_global.groupby('ID_sujeto')[col].shift(1).fillna(0)
            # Lag 2 minutos (t-2)
            df_maestro_global[f'{col}_lag_2m_norm'] = df_maestro_global.groupby('ID_sujeto')[col].shift(2).fillna(0)
            # Lag 3 minutos (t-3)
            df_maestro_global[f'{col}_lag_3m_norm'] = df_maestro_global.groupby('ID_sujeto')[col].shift(3).fillna(0)

       # --------------------------------------------------------------------------------------------------
    # PASO 5: ALMACENAMIENTO DEL DATASET MAESTRO DEFINITIVO
    # --------------------------------------------------------------------------------------------------
    # ruta_salida_final = f"WESAD_features_maestro_{modo_frecuencia}.csv"
    # df_maestro_global.to_csv(ruta_salida_final, index=False)
    
    print(f"\n💾 [PROCESO COMPLETADO] Dataset consolidado con éxito.")
    # print(f"   --> Archivo: {ruta_salida_final}")
    print(f"   --> Dimensiones finales de la matriz para ML: {df_maestro_global.shape}")


🧱 INICIANDO GENERACIÓN DEL DATASET MAESTRO EN FRECUENCIA: [1/60hz]

🚀 === PROCESANDO SUJETO: S2 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 07:15:25 | HR: 07:15:35).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7864


C:\Users\miner\AppData\Local\Temp\ipykernel_134104\1118443529.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)


   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1535s | Error de ajuste: 0.0000°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S3 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 11:08:48 | HR: 11:08:58).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7712


C:\Users\miner\AppData\Local\Temp\ipykernel_134104\1118443529.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unido['Boton_Pulsado'] = df_unido['Boton_Pulsado'].fillna(0).astype(int)


   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1080s | Error de ajuste: 0.0066°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S4 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 08:33:40 | HR: 08:33:50).
   [✓] Checkpoint 3: Señales continuas remuestreadas a 1Hz.
   [✓] Checkpoint 4: Eventos discretos (IBI exacto y Tags) calculados.
   [✓] Checkpoint 5: Fusión a 1 Hz pura realizada. Filas base: 7988
   [✓] Checkpoint 6: Sincronización por curva térmica completada. 
       -> Segundo de inicio (Offset): 1465s | Error de ajuste: 0.0063°C
🚀 Extrayendo características y colapsando etiquetas en modo [1hz]...

🚀 === PROCESANDO SUJETO: S5 [Modo: 1hz] ===
   [✓] Checkpoint 1: Rutas y archivos esenciales verificados.
   [✓] Checkpoint 2: Relojes base sincronizados (ACC: 12:22:13 | HR: 12:22:23).
   [✓] Chec

In [26]:
# ======================================================================================================
# 💾  ALMACENAMIENTO PERMANENTE DEL DATASET MAESTRO DE 1 MINUTO
# ======================================================================================================
from pathlib import Path

# Extraemos o definimos nuevas rutas específicas para los archivos agregados de ML
# Extraemos las rutas directamente del archivo de configuración JSON
ruta_salida_parquet_1min = Path(config["rutas_procesamiento"]["ruta_salida_parquet_1min"])
ruta_salida_csv_1min = Path(config["rutas_procesamiento"]["ruta_salida_csv_1min"])


# Nos aseguramos de que la carpeta de destino exista en el disco
ruta_salida_parquet_1min.parent.mkdir(parents=True, exist_ok=True)

print(f"\n💾 Guardando Dataset definitivo de ML con {df_maestro_global.shape[1]} columnas...")

# Guardamos en formato Parquet usando fastparquet para evitar el bug de PyArrow
df_maestro_global.to_parquet(ruta_salida_parquet_1min, engine='fastparquet', index=False)
print(f"   [✓] Guardado con éxito en formato Parquet: {ruta_salida_parquet_1min}")

# Guardamos en formato CSV para inspección o cargas rápidas externas
df_maestro_global.to_csv(ruta_salida_csv_1min, index=False, encoding='utf-8')
print(f"   [✓] Guardado con éxito en formato CSV: {ruta_salida_csv_1min}")

print(f"\n🎉 ¡PROCESO DE INGENIERÍA PARA DATASET RESUMIDO (1 MINUTO) TERMINADO! 🎉")
print(f"Dimensiones de la matriz lista para entrenar: {df_maestro_global.shape}")


💾 Guardando Dataset definitivo de ML con 89 columnas...
   [✓] Guardado con éxito en formato Parquet: ..\data\processed\WESAD_dataset_maestro_1min.parquet
   [✓] Guardado con éxito en formato CSV: ..\data\processed\WESAD_dataset_maestro_1min.csv

🎉 ¡PROCESO DE INGENIERÍA PARA DATASET RESUMIDO (1 MINUTO) TERMINADO! 🎉
Dimensiones de la matriz lista para entrenar: (1425, 89)


El pipeline de agregación genera un dataset final de 1.425 filas y 89 columnas, resultado de reducir el registro de 1Hz a resolución de 1 minuto, enriquecerlo con ventanas de contexto de 3, 5 y 10 minutos, aplicar la normalización Z-score por baseline y añadir los lags temporales. Este dataset queda almacenado tanto en formato Parquet como CSV, y es la matriz definitiva sobre la que se entrena `StressLoad(t)`.

## 2.3. Auditoría de calidad y control de coherencia metodológica

Tras generar el dataset maestro a resolución de 1 minuto, se ejecuta una batería de pruebas automáticas de control de calidad, extendidas a la totalidad de los 15 sujetos y no a una muestra puntual:

- **Prueba 1 — Integridad estructural:** ausencia de valores nulos, desglosada por sujeto.
- **Prueba 2 — Normalización Z-score baseline-referenced:** corrección matemática de la tipificación fisiológica person-specific.
- **Prueba 3 — Coherencia temporal:** propagación correcta de las variables de lag y continuidad de la secuencia de minutos.
- **Prueba 4 — Distribución de la variable objetivo:** balance y representación de clases entre sujetos.

In [27]:
# 📋 PRUEBA 1 (extendida): Integridad estructural por sujeto
print(f"📋 PRUEBA 1 - Integridad de la matriz (desglose por sujeto):")
print(f"   -> Dimensiones totales: {df_maestro_global.shape[0]} filas x {df_maestro_global.shape[1]} columnas.")

nulos_por_sujeto = df_maestro_global.groupby('ID_sujeto').apply(
    lambda g: g.isnull().sum().sum(), include_groups=False
)
filas_por_sujeto = df_maestro_global.groupby('ID_sujeto').size()

df_integridad = pd.DataFrame({
    'n_filas': filas_por_sujeto,
    'n_nulos': nulos_por_sujeto
})
print(df_integridad.to_string())

if df_integridad['n_nulos'].sum() == 0:
    print("   ✅ CORRECTO: Ningún sujeto presenta valores nulos.")
else:
    sujetos_con_nulos = df_integridad[df_integridad['n_nulos'] > 0].index.tolist()
    print(f"   ❌ ALERTA: Nulos detectados en sujetos: {sujetos_con_nulos}")

📋 PRUEBA 1 - Integridad de la matriz (desglose por sujeto):
   -> Dimensiones totales: 1425 filas x 89 columnas.
           n_filas  n_nulos
ID_sujeto                  
S10             91        0
S11             86        0
S13             90        0
S14             92        0
S15             87        0
S16             93        0
S17             98        0
S2             101        0
S3             107        0
S4             106        0
S5              96        0
S6             115        0
S7              87        0
S8              91        0
S9              85        0
   ✅ CORRECTO: Ningún sujeto presenta valores nulos.


In [28]:
# ⚖️ PRUEBA 2: Validación Matemática del Z-Score Personalizado (baseline-referenced)
# Extendida a TODOS los sujetos del dataset maestro, no solo un spot check.
col_test = "HR_mean_norm"
label_baseline = 1
tolerancia = 1e-4

print(f"\n⚖️ PRUEBA 2 - Filtro Z-Score baseline-referenced ({col_test}), todos los sujetos:")

resultados_auditoria = []
for sujeto in sorted(df_maestro_global['ID_sujeto'].unique()):
    df_suj_baseline = df_maestro_global[
        (df_maestro_global['ID_sujeto'] == sujeto) &
        (df_maestro_global['Etiqueta'] == label_baseline)
    ]
    n_baseline = len(df_suj_baseline)

    if n_baseline == 0:
        resultados_auditoria.append({
            'ID_sujeto': sujeto, 'n_baseline': 0,
            'media': np.nan, 'std': np.nan, 'estado': '⚠️ SIN BASELINE'
        })
        continue

    media_z = df_suj_baseline[col_test].mean()
    std_z = df_suj_baseline[col_test].std()

    if n_baseline < 5:
        estado = '⚠️ MUESTRA PEQUEÑA (n<5)'
    elif abs(media_z) < tolerancia and abs(std_z - 1.0) < tolerancia:
        estado = '✅ CORRECTO'
    else:
        estado = '❌ ALERTA'

    resultados_auditoria.append({
        'ID_sujeto': sujeto, 'n_baseline': n_baseline,
        'media': media_z, 'std': std_z, 'estado': estado
    })

df_auditoria_zscore = pd.DataFrame(resultados_auditoria)
print(df_auditoria_zscore.to_string(index=False))

# Resumen agregado
n_correctos = (df_auditoria_zscore['estado'] == '✅ CORRECTO').sum()
n_total = len(df_auditoria_zscore)
print(f"\n   -> Sujetos validados correctamente: {n_correctos}/{n_total}")
if df_auditoria_zscore['n_baseline'].min() < 5:
    print(f"   -> Mínimo de minutos de baseline observado: {df_auditoria_zscore['n_baseline'].min()} "
          f"(sujeto {df_auditoria_zscore.loc[df_auditoria_zscore['n_baseline'].idxmin(), 'ID_sujeto']})")


⚖️ PRUEBA 2 - Filtro Z-Score baseline-referenced (HR_mean_norm), todos los sujetos:
ID_sujeto  n_baseline         media  std     estado
      S10          19  1.358562e-16  1.0 ✅ CORRECTO
      S11          20  1.387779e-15  1.0 ✅ CORRECTO
      S13          20  9.103829e-16  1.0 ✅ CORRECTO
      S14          19 -1.747140e-15  1.0 ✅ CORRECTO
      S15          19 -1.040104e-15  1.0 ✅ CORRECTO
      S16          19 -1.402387e-15  1.0 ✅ CORRECTO
      S17          19 -2.804774e-16  1.0 ✅ CORRECTO
       S2          19 -3.330669e-16  1.0 ✅ CORRECTO
       S3          19  9.349247e-17  1.0 ✅ CORRECTO
       S4          19 -6.369174e-16  1.0 ✅ CORRECTO
       S5          20  3.275158e-16  1.0 ✅ CORRECTO
       S6          19 -1.633197e-15  1.0 ✅ CORRECTO
       S7          19 -5.609548e-16  1.0 ✅ CORRECTO
       S8          19  7.523222e-16  1.0 ✅ CORRECTO
       S9          20  8.229528e-16  1.0 ✅ CORRECTO

   -> Sujetos validados correctamente: 15/15


In [29]:
# ⏳ PRUEBA 3 (extendida): Verificación sistemática de la propiedad de lag, todos los sujetos
print(f"\n⏳ PRUEBA 3 - Verificación sistemática de inercia temporal (lag 1 minuto):")

df_ordenado = df_maestro_global.sort_values(['ID_sujeto', 'minuto_id']).reset_index(drop=True)
incoherencias = []

for sujeto, grupo in df_ordenado.groupby('ID_sujeto'):
    grupo = grupo.reset_index(drop=True)
    # El valor de HR_mean_norm en t debe coincidir con HR_mean_norm_lag_1m_norm en t+1
    valor_actual = grupo['HR_mean_norm'].iloc[:-1].values
    valor_lag_siguiente = grupo['HR_mean_norm_lag_1m_norm'].iloc[1:].values

    # Comparación con tolerancia de punto flotante
    desacuerdos = ~np.isclose(valor_actual, valor_lag_siguiente, equal_nan=True, atol=1e-8)
    n_desacuerdos = desacuerdos.sum()

    if n_desacuerdos > 0:
        incoherencias.append((sujeto, n_desacuerdos, len(grupo)))

if len(incoherencias) == 0:
    print(f"   ✅ CORRECTO: La propiedad de lag se cumple en los {df_ordenado['ID_sujeto'].nunique()} sujetos.")
else:
    print(f"   ❌ ALERTA: Incoherencias de lag detectadas en {len(incoherencias)} sujeto(s):")
    for sujeto, n_desacuerdos, n_total in incoherencias:
        print(f"      -> {sujeto}: {n_desacuerdos}/{n_total} minutos discrepantes")

# Verificación adicional: continuidad de minuto_id por sujeto (sin huecos)
print(f"\n⏳ PRUEBA 3b - Continuidad de minuto_id (sin huecos temporales):")
huecos = []
for sujeto, grupo in df_ordenado.groupby('ID_sujeto'):
    minutos = sorted(grupo['minuto_id'].unique())
    esperado = list(range(minutos[0], minutos[-1] + 1))
    if minutos != esperado:
        huecos.append(sujeto)

if len(huecos) == 0:
    print("   ✅ CORRECTO: Ningún sujeto presenta huecos en la secuencia de minutos.")
else:
    print(f"   ❌ ALERTA: Huecos temporales detectados en: {huecos}")


⏳ PRUEBA 3 - Verificación sistemática de inercia temporal (lag 1 minuto):
   ✅ CORRECTO: La propiedad de lag se cumple en los 15 sujetos.

⏳ PRUEBA 3b - Continuidad de minuto_id (sin huecos temporales):
   ✅ CORRECTO: Ningún sujeto presenta huecos en la secuencia de minutos.


In [30]:
# 🎯 PRUEBA 4 (extendida): Distribución de clases por sujeto
print(f"\n🎯 PRUEBA 4 - Distribución de la variable objetivo por sujeto:")
tabla_clases = pd.crosstab(df_maestro_global['ID_sujeto'], df_maestro_global['Etiqueta'])
print(tabla_clases.to_string())

# Sujetos que no aportan ninguna muestra de alguna clase presente en el global
clases_globales = set(df_maestro_global['Etiqueta'].unique())
print(f"\n   -> Clases ausentes por sujeto:")
for sujeto in tabla_clases.index:
    clases_sujeto = set(tabla_clases.columns[tabla_clases.loc[sujeto] > 0])
    ausentes = clases_globales - clases_sujeto
    if ausentes:
        print(f"      -> {sujeto}: no presenta clase(s) {sorted(ausentes)}")


🎯 PRUEBA 4 - Distribución de la variable objetivo por sujeto:
Etiqueta    0   1   2  3   4  5  6  7
ID_sujeto                            
S10        37  19  12  6  14  1  1  1
S11        34  20  11  6  13  1  1  0
S13        38  20  11  6  14  1  0  0
S14        40  19  11  6  13  1  1  1
S15        35  19  11  6  13  1  1  1
S16        42  19  11  6  12  1  1  1
S17        47  19  12  6  12  1  1  0
S2         51  19  10  6  13  0  1  1
S3         54  19  11  7  12  1  1  2
S4         54  19  11  6  13  1  1  1
S5         45  20  11  6  12  1  0  1
S6         63  19  11  6  14  1  0  1
S7         37  19  10  6  12  1  1  1
S8         41  19  11  6  12  1  0  1
S9         31  20  11  6  14  1  1  1

   -> Clases ausentes por sujeto:
      -> S11: no presenta clase(s) [np.int64(7)]
      -> S13: no presenta clase(s) [np.int64(6), np.int64(7)]
      -> S17: no presenta clase(s) [np.int64(7)]
      -> S2: no presenta clase(s) [np.int64(5)]
      -> S5: no presenta clase(s) [np.int64(6)]


Tras la generación del dataset maestro (1425 filas × 89 columnas, 15 sujetos), se ejecutó una batería de pruebas automáticas de control de calidad, extendida a la totalidad de la muestra (no a un *spot check* puntual), orientada a verificar la integridad estructural, la correcta normalización fisiológica por sujeto, la coherencia temporal de las variables derivadas y el balance de la variable objetivo.

### Prueba 1 — Integridad estructural

Se verificó la ausencia de valores nulos de forma desglosada por sujeto, no solo a nivel global. Los 15 sujetos presentan cero valores nulos en las 89 columnas, con tamaños muestrales por sujeto entre 85 filas (S9) y 115 filas (S6) — una dispersión atribuible a la duración variable de las sesiones de grabación de WESAD, sin ningún sujeto anómalamente incompleto.

### Prueba 2 — Normalización Z-score baseline-referenced

La estrategia de normalización adoptada calcula la media y desviación típica de cada variable fisiológica exclusivamente sobre los minutos de estado basal (baseline) de cada sujeto, y aplica esa tipificación a la totalidad de su serie temporal. Esta decisión metodológica expresa la activación fisiológica como desviación respecto al reposo individual, coherente con el objetivo de inferir `StressLoad(t)` de forma person-specific, en lugar de emplear un Z-score poblacional que mezclaría la variabilidad de reposo y de activación en una única vara de medida.

La verificación, extendida a los 15 sujetos, confirma que en todos los casos la media del baseline normalizado es ≈0.0 (del orden de 1e-15 a 1e-16, error de precisión de punto flotante) y la desviación típica es exactamente 1.0. El número de minutos de baseline disponibles por sujeto es homogéneo (19–20 minutos), lo que respalda la fiabilidad de las estimaciones de σ utilizadas para la tipificación y descarta el riesgo de normalización basada en muestras pequeñas y ruidosas.

### Prueba 3 — Coherencia temporal

Se verificó sistemáticamente, sobre los 15 sujetos y la totalidad de las 1425 filas, que el valor de cada variable normalizada en el minuto *t* se propaga correctamente a su correspondiente columna de lag en el minuto *t+1*, confirmando la inercia temporal esperada en las features autorregresivas. Adicionalmente, se verificó la continuidad de la secuencia `minuto_id` por sujeto, confirmando la ausencia de huecos temporales (ningún minuto perdido dentro del rango de grabación de cada sujeto).

### Prueba 4 — Distribución de la variable objetivo

El desglose por sujeto mediante tabla de contingencia (`ID_sujeto` × `Etiqueta`) confirma que las clases mayoritarias utilizadas en el modelado (0: transición/no definida, 1: baseline, 2: estrés, 3: amusement, 4: meditación) están representadas de forma consistente en los 15 sujetos, con las siguientes observaciones:

- **Clase 1 (baseline):** 19–20 minutos por sujeto, homogénea — coherente con lo observado en la Prueba 2.
- **Clase 3 (amusement):** prácticamente constante en 6 minutos por sujeto (7 en el caso de S3), sugiriendo una condición experimental de duración fija en el protocolo original de WESAD.
- **Clase 0:** la más heterogénea entre sujetos (31–63 minutos), atribuible a su naturaleza de estado transicional/no definido.
- **Clases minoritarias (5, 6, 7):** además de su baja representación porcentual global (<1% cada una), están **ausentes en 7 de los 15 sujetos** (S11, S13, S17, S2, S5, S6, S8 carecen de al menos una de las tres). Esta inconsistencia de representación entre sujetos, y no solo su bajo volumen, es el argumento principal para su exclusión del conjunto de clases modeladas en este TFM.

### Conclusión general

La auditoría extendida a los 15 sujetos confirma la integridad estructural, la correcta tipificación fisiológica person-specific, la coherencia temporal y la trazabilidad del balance de clases del dataset maestro. No se identificaron incidencias reales tras la corrección de la Prueba 2; el dataset se considera validado y apto para la siguiente etapa del pipeline (modelado de `StressLoad(t)`).